# Task09 — Mở rộng sang 25% và 50% (`52` lượt `main`, `phase2-v1` / `colab-task06-v1`)

Chọn **Runtime → Change runtime type → GPU** (ưu tiên **`NVIDIA A100-SXM4-80GB`** để đồng nhất phần cứng và đối sánh trực tiếp `train_seconds` / `gpu_hours` với Task 08 trên toàn bộ **78 cấu hình lõi `10%–25%–50%`**; vẫn tương thích **`NVIDIA L4`**), kiểm tra các đường dẫn đã điền sẵn ở **Bước 1**, rồi chạy lần lượt từng cell theo thứ tự.

### Phạm vi & Thứ tự ưu tiên (`cut_order` theo `configs/phase2.yaml`)
- **Mục tiêu:** Chạy trọn vẹn **52 lượt huấn luyện chính (`run_mode="main"`, backbone `ViSoBERT`, `BF16 strict`)** ở hai mức dữ liệu đích:
  1. **Mức `25%` (ưu tiên chạy trọn vẹn trước — 26 lượt):** `SA` (`5.762` câu `train`, `181` step/ep) `5 DFT → 3 C0 → 5 Seq` + `SB` (`1.386` câu `train`, `44` step/ep) `5 DFT → 3 C0 → 5 Seq`.
  2. **Mức `50%` (chạy tiếp sau — 26 lượt):** `SA` (`11.524` câu `train`, `361` step/ep) `5 DFT → 3 C0 → 5 Seq` + `SB` (`2.771` câu `train`, `87` step/ep) `5 DFT → 3 C0 → 5 Seq`.
- **Tách phạm vi có chủ đích:** Không chạy mức `5%` (`OPTIONAL_DEFERRED`), và không chạy `10` lượt `100% SA` trong phiên này (chạy riêng trước/cùng Task 10). Mốc `100%` Giai đoạn 1 chỉ giữ vai trò **tham chiếu lịch sử** (khác giao thức).
- **LR khóa từ Task 07 (`selection_p1_lr.json`):** `SA/dft = 3e-5`, `SA/c0 = SA/seq = 2e-5`; `SB/dft = 2e-5` (`grid_edge=True`), `SB/c0 = SB/seq = 3e-5` (`grid_edge=True`). Không thay đổi LR hay lưới sau khi mở `test`.

### Lưu ý vận hành, Dung lượng & Phục hồi xuyên phiên
- **Dự toán thực tế (từ số đo `A100` ở Task 08 `0.4400h`):** 52 lượt hết khoảng **`2,0 – 3,3` GPU-giờ** trên `A100` (`25%` ~`0,7–1,1h`, `50%` ~`1,3–2,2h`).
- **Dung lượng Drive:** 20 checkpoint `DFT` (`25%` + `50%`) cần **`~7,8 GB`** trên Drive nếu `BACKUP_DFT_CKPTS_TO_DRIVE="all"`. Bước 2 tự kiểm tra dung lượng trống `/content/drive` để cảnh báo sớm (có thể dọn `Task08/checkpoints` không còn dùng hoặc chuyển sang `"c0_only"`/`"none"`).
- **Đường cong `10% → 25% → 50%`:** Bước 10 tự đọc thêm kết quả `10%` ở `/content/drive/MyDrive/Xdomain/Phase2/Task08` (nếu có sẵn) để tổng hợp bảng 78 cấu hình; toàn bộ đối chứng ở `25%` và `50%` là **thăm dò (`exploratory=True`, `p_holm=NaN`)** — giữ bất biến gia đình `Holm m=2` tại `10%`.


## 1. Tham số, profile môi trường và đường dẫn cố định
Khai báo tập trung cấu hình phiên chạy Task 09, thứ tự mức `FRACTIONS_TO_RUN = (25, 50)`, và các đường dẫn Google Drive kế thừa từ Task 07–08 (`SESSION_ID` chỉ gắn vào tên file log/môi trường để giữ nguyên `fingerprint`).


In [ ]:
from pathlib import Path
import csv, hashlib, json, os, platform, re, shutil, signal, subprocess, sys, time

T0_SETUP = time.perf_counter()
SESSION_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())

REPO_URL = "https://github.com/PhuongNgoVan/Xdomain-Transfer-Learning.git"
REPO_REF = "phase2-resource-transfer"  # đổi sang commit SHA 40 ký tự nếu nối lại phiên cũ sau khi nhánh có commit mới
PRIVATE_REPO = False

PROFILE_ID = "colab-task06-v1"
PROFILES = {
    "colab-task06-v1": {
        "python": "3.13.18",
        "torch": "2.14.1+cu130",
        "cuda": "13.0",
        "pins": {
            "transformers": "4.57.1",
            "huggingface-hub": "0.36.2",
            "tokenizers": "0.22.2",
            "numpy": "2.1.3",
            "pandas": "3.1.0",
            "scikit-learn": "1.6.1",
            "scipy": "1.16.3",
            "PyYAML": "6.0.2",
            "sentencepiece": "0.2.2",
            "protobuf": "7.36.2",
            "safetensors": "0.8.0",
            "tqdm": "4.67.3",
        },
    },
}
assert PROFILE_ID in PROFILES, f"Unknown PROFILE_ID: {PROFILE_ID}"

FRACTIONS_TO_RUN = (25, 50)  # chạy toàn bộ 25% trước -> 50% sau; đổi thành (25,) nếu cần cắt 50% theo cut_order
assert tuple(FRACTIONS_TO_RUN) in ((25, 50), (25,)), f"FRACTIONS_TO_RUN không hợp lệ: {FRACTIONS_TO_RUN}"

MOUNT_DRIVE = True
# Nguồn 10 checkpoint Stage 1 (ViSoBERT, seed 0..4) trên Drive (điền sẵn theo Task 07-08):
DRIVE_SOURCE_CKPT_ROOT = ""
DRIVE_VSMEC_CKPT_DIR = "/content/drive/MyDrive/Xdomain/finetune-vsmec-7c/visobert/ckpt_baseline"
DRIVE_NEUESC_CKPT_DIR = "/content/drive/MyDrive/Xdomain/finetune-neu-3c/visobert/ckpt_baseline"
# Nguồn selection_p1_lr.json đã khóa từ Task 07:
DRIVE_SELECTION_JSON_PATH = "/content/drive/MyDrive/Xdomain/Phase2/Task07/selection_p1_lr.json"
# Kết quả Task 08 (10%) trên Drive (đọc chỉ-đọc để ghép đường cong 10% -> 25% -> 50% ở Bước 10):
DRIVE_TASK08_DIR = Path("/content/drive/MyDrive/Xdomain/Phase2/Task08")
INCLUDE_TASK08_10PCT_IN_SCALING = True

USE_DRIVE_BACKUP = True
DRIVE_BACKUP_BASE = Path("/content/drive/MyDrive/Xdomain/Phase2/Task09")
BACKUP_DFT_CKPTS_TO_DRIVE = "all"  # "all" (~7.8 GB cho 20 DFT) | "c0_only" (~4.68 GB) | "none"
assert BACKUP_DFT_CKPTS_TO_DRIVE in ("all", "c0_only", "none")
SAVE_ALL_MAIN_CKPTS = False        # False: chỉ lưu checkpoint 20 DFT (cần cho C0) và truyền --no-save-ckpt cho 32 C0/Seq
CLEAN_COMPLETED_RESUME_PT = True   # xóa work/<run_id>/resume.pt (~1.1 GB) sau complete=True; giữ provenance.json

assert Path("/content").is_dir(), "Notebook này yêu cầu môi trường Google Colab (/content)."
REPO_DIR = Path("/content/xdomain-task09")
VENV_DIR = Path(f"/content/phase2-task09-{PROFILE_ID}-venv")
SOURCE_CKPT_STAGE_DIR = Path("/content/phase2_source_ckpts")
assert "sweep" not in str(SOURCE_CKPT_STAGE_DIR).lower() and "smoke" not in str(SOURCE_CKPT_STAGE_DIR).lower(), (
    "SOURCE_CKPT_STAGE_DIR không được chứa 'sweep' hoặc 'smoke' (src/zs/ckpt.py sẽ loại bỏ)."
)
SELECTION_STAGE_DIR = Path("/content/phase2_locked_selection")
SELECTION_STAGE_PATH = SELECTION_STAGE_DIR / "selection_p1_lr.json"
OUTPUT_BASE = Path("/content/task09_outputs")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print(
    f"Session: {SESSION_ID} | Profile: {PROFILE_ID} | Fractions: {FRACTIONS_TO_RUN} | "
    f"DFT Drive backup: {BACKUP_DFT_CKPTS_TO_DRIVE}"
)


## 2. Kiểm tra GPU CUDA/BF16 ở kernel, mount Drive & kiểm tra dung lượng trống
Kiểm tra phần cứng và phiên bản Torch/CUDA (giữ 0 MiB VRAM ở kernel và không `assert` Python version ở kernel vì kernel Colab `3.13.16` khác Python hệ thống `3.13.18`), mount Google Drive, xác nhận đường dẫn checkpoint nguồn và kiểm tra dung lượng trống trên `/content` và `/content/drive`.


In [ ]:
import torch

assert torch.cuda.is_available(), "Chưa bật GPU CUDA trên Colab (Runtime -> Change runtime type -> GPU)."
assert torch.cuda.is_bf16_supported(), "GPU hiện tại không hỗ trợ BF16 strict; hãy chọn A100 hoặc L4."

gpu_name = torch.cuda.get_device_name(0)
vram_gib = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
print(f"GPU: {gpu_name} | VRAM (GiB): {vram_gib}")
if "A100" not in gpu_name:
    print(f"NOTE: Task 08 chạy trên NVIDIA A100-SXM4-80GB; bạn đang dùng {gpu_name} — nhớ ghi rõ khi đối chiếu thời gian/VRAM.")
print(
    f"Kernel Python: {platform.python_version()} | "
    f"Preinstalled Torch: {torch.__version__} | CUDA: {torch.version.cuda} | BF16: True"
)

expected_profile = PROFILES[PROFILE_ID]
assert torch.__version__ == expected_profile["torch"], (
    f"Torch trên Colab ({torch.__version__}) khác profile {PROFILE_ID} ({expected_profile['torch']})."
)
assert torch.version.cuda == expected_profile["cuda"], (
    f"CUDA của Torch ({torch.version.cuda}) khác profile {PROFILE_ID} ({expected_profile['cuda']})."
)
subprocess.run(["nvidia-smi"], check=True)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if DRIVE_SOURCE_CKPT_ROOT.strip():
    root_p = Path(DRIVE_SOURCE_CKPT_ROOT.strip())
    CKPT_SEARCH_ROOTS = {"vsmec": root_p, "neu3": root_p}
elif DRIVE_VSMEC_CKPT_DIR.strip() and DRIVE_NEUESC_CKPT_DIR.strip():
    CKPT_SEARCH_ROOTS = {
        "vsmec": Path(DRIVE_VSMEC_CKPT_DIR.strip()),
        "neu3": Path(DRIVE_NEUESC_CKPT_DIR.strip()),
    }
else:
    raise RuntimeError("Chưa điền đường dẫn checkpoint nguồn Stage 1 (seed 0..4) ở Bước 1.")

for dom, folder in CKPT_SEARCH_ROOTS.items():
    if not folder.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục checkpoint nguồn ({dom}): {folder}")
    print(f"Verified source search root ({dom}): {folder}")

ssd_free_gib = round(shutil.disk_usage("/content").free / (1024 ** 3), 2)
print(f"/content SSD free space: {ssd_free_gib} GiB")
assert ssd_free_gib >= 16.0, f"Không gian /content còn quá thấp ({ssd_free_gib} GiB < 16 GiB)."

if MOUNT_DRIVE and Path("/content/drive/MyDrive").is_dir():
    drive_free_gib = round(shutil.disk_usage("/content/drive").free / (1024 ** 3), 2)
    print(f"Google Drive free space: {drive_free_gib} GiB")
    if USE_DRIVE_BACKUP and BACKUP_DFT_CKPTS_TO_DRIVE == "all" and drive_free_gib < 10.0:
        print(
            f"WARNING: Google Drive chỉ còn {drive_free_gib} GiB; backup all 20 checkpoint DFT cần ~7.8 GB. "
            "Cân nhắc dọn Task08/checkpoints sau khi đã xong Task 08 hoặc đổi BACKUP_DFT_CKPTS_TO_DRIVE sang 'c0_only'/'none'."
        )
if INCLUDE_TASK08_10PCT_IN_SCALING:
    print("Task 08 (10%) dir exists on Drive:", DRIVE_TASK08_DIR.is_dir(), "| path:", DRIVE_TASK08_DIR)


## 3. Fetch repository, cố định commit, stage `selection_p1_lr.json`, chụp `frozen_before.json` & `hydrate` từ Drive phẳng
Fetch `REPO_REF`, checkout `detached HEAD`, kiểm tra đủ 30 file chỉ mục `f{10,25,50}_s{0..4}_ids.csv`, stage nguyên byte `selection_p1_lr.json`, lưu bảng băm tài nguyên đóng băng, kiểm tra chốt chặn `commit`/`profile_id` trên Drive phẳng và tự `hydrate` các lượt đã `complete` (kèm checkpoint `DFT`) khi nối phiên.


In [ ]:
import base64

def sha256_path(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_copy_file(src: Path, dst: Path) -> None:
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + f".{os.getpid()}.tmp")
    shutil.copy2(src, tmp)
    tmp.replace(dst)

def git_command(args, cwd=None) -> str:
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    if PRIVATE_REPO:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        assert token, "Thiếu Colab Secret GITHUB_TOKEN khi PRIVATE_REPO=True."
        auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
        env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.extraHeader",
            GIT_CONFIG_VALUE_0="Authorization: Basic " + auth,
        )
    result = subprocess.run(
        ["git", *args], cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Git thất bại: kiểm tra URL, quyền đọc hoặc commit SHA đã được push chưa.\n" + result.stderr
        )
    return result.stdout.strip()

def hydrate_from_flat_drive(
    drive_root: Path,
    results_dir: Path,
    ckpt_out_dir: Path,
    commit: str,
    profile_id: str,
    allowed_fractions=(25, 50),
) -> set[str]:
    # Check flat Drive safety (commit/profile/fractions) and hydrate complete runs + DFT checkpoints.
    drive_env_json = drive_root / "environment" / "environment.json"
    if drive_env_json.is_file():
        d_env = json.loads(drive_env_json.read_text(encoding="utf-8"))
        assert d_env.get("commit") == commit, (
            f"Thư mục Drive phẳng {drive_root} thuộc commit khác ({d_env.get('commit')} != {commit})."
        )
        assert d_env.get("profile_id") == profile_id, (
            f"Thư mục Drive phẳng {drive_root} thuộc profile khác ({d_env.get('profile_id')} != {profile_id})."
        )

    drive_csv = drive_root / "phase2_runs.csv"
    if not drive_csv.is_file():
        return set()

    with drive_csv.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        rows = list(reader)
    if not rows:
        return set()

    allowed_set = {str(int(f)) for f in allowed_fractions}
    commits_in_csv = {r["git_commit"] for r in rows}
    assert commits_in_csv <= {commit}, f"Backup trên Drive thuộc commit khác: {commits_in_csv} != {commit}"
    assert all(r["run_mode"] == "main" and str(r["fraction"]) in allowed_set for r in rows), (
        f"Backup trên Drive chứa dòng ngoài Task 09 (main, fraction in {sorted(allowed_set)})."
    )

    if (results_dir / "phase2_runs.csv").is_file():
        return set()

    verified_run_ids: set[str] = set()
    for run_id in sorted({r["run_id"] for r in rows}):
        d_run = drive_root / "runs" / run_id
        d_prov = drive_root / "work" / run_id / "provenance.json"
        d_comp = d_run / "completion.json"
        if not (d_comp.is_file() and d_prov.is_file()):
            continue
        comp = json.loads(d_comp.read_text(encoding="utf-8"))
        prov = json.loads(d_prov.read_text(encoding="utf-8"))
        if comp.get("complete") is not True or comp.get("fingerprint") != prov.get("fingerprint"):
            continue

        artifacts = comp.get("artifacts", {})
        ckpt_rel_prefix = comp.get("ckpt_path")
        ckpt_files_expected = list(comp.get("checkpoint_files") or [])

        copy_plan: list[tuple[Path, Path]] = []
        ok = bool(artifacts)
        if ckpt_rel_prefix is not None:
            ckpt_prefix = str(ckpt_rel_prefix).rstrip("/") + "/"
            if not ckpt_files_expected:
                ok = False
        else:
            ckpt_prefix = ""

        covered_ckpt_files: set[str] = set()
        for rel, digest in artifacts.items():
            if not rel.startswith("../"):
                src_f = d_run / rel
                dst_f = results_dir / "runs" / run_id / rel
            elif ckpt_prefix and rel.startswith(ckpt_prefix):
                sub_rel = rel[len(ckpt_prefix):]
                covered_ckpt_files.add(sub_rel)
                src_f = drive_root / "checkpoints" / run_id / sub_rel
                dst_f = ckpt_out_dir / run_id / sub_rel
            else:
                ok = False
                break
            if not src_f.is_file() or sha256_path(src_f) != digest:
                ok = False
                break
            copy_plan.append((src_f, dst_f))

        if ok and ckpt_rel_prefix is not None and not set(ckpt_files_expected).issubset(covered_ckpt_files):
            ok = False

        if not ok:
            print(f"Skipping hydrate for {run_id} (incomplete artifacts/checkpoint on Drive; will recompute cleanly).")
            continue

        atomic_copy_file(d_prov, results_dir / "work" / run_id / "provenance.json")
        for src_f, dst_f in copy_plan:
            atomic_copy_file(src_f, dst_f)
        atomic_copy_file(d_comp, results_dir / "runs" / run_id / "completion.json")
        verified_run_ids.add(run_id)

    if verified_run_ids:
        kept_rows = [r for r in rows if r["run_id"] in verified_run_ids]
        tmp_csv = results_dir / "phase2_runs.csv.tmp"
        with tmp_csv.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames, lineterminator="\n")
            writer.writeheader()
            writer.writerows(kept_rows)
        tmp_csv.replace(results_dir / "phase2_runs.csv")
        print(f"Hydrated {len(verified_run_ids)} verified Task09 main runs from flat Drive backup.")
    return verified_run_ids

if not REPO_DIR.exists():
    git_command(["clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)])
if not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"REPO_DIR tồn tại nhưng không phải Git checkout: {REPO_DIR}")
if not git_command(["ls-files"], REPO_DIR) and all(p.name == ".git" for p in REPO_DIR.iterdir()):
    git_command(["restore", "--source=HEAD", "--staged", "--worktree", "--", "."], REPO_DIR)
if git_command(["diff", "--name-only"], REPO_DIR) or git_command(["diff", "--cached", "--name-only"], REPO_DIR):
    raise RuntimeError("Checkout có thay đổi cục bộ; hãy chọn REPO_DIR sạch.")

git_command(["fetch", "--depth=1", "origin", REPO_REF], REPO_DIR)
git_command(["checkout", "--detach", "FETCH_HEAD"], REPO_DIR)
COMMIT = git_command(["rev-parse", "HEAD"], REPO_DIR)
assert len(COMMIT) == 40, f"Commit SHA không hợp lệ: {COMMIT}"

OUTPUT_DIR = OUTPUT_BASE / COMMIT[:12] / PROFILE_ID
RESULTS_DIR = OUTPUT_DIR / "results" / "phase2"
CKPT_OUT_DIR = OUTPUT_DIR / "checkpoints" / "phase2"
ENV_DIR = OUTPUT_DIR / "environment"
LOG_DIR = OUTPUT_DIR / "logs"
for folder in (RESULTS_DIR, CKPT_OUT_DIR, ENV_DIR, LOG_DIR, SELECTION_STAGE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DRIVE_SESSION_ROOT = DRIVE_BACKUP_BASE if (MOUNT_DRIVE and USE_DRIVE_BACKUP) else None
if DRIVE_SESSION_ROOT is not None:
    DRIVE_SESSION_ROOT.mkdir(parents=True, exist_ok=True)

required_repo_files = [
    "configs/phase2.yaml",
    "src/phase2/cli.py",
    "src/phase2/run.py",
    "src/phase2/models.py",
    "src/phase2/protocol.py",
    "src/phase2/subsets.py",
    "src/seq/analysis.py",
    "src/zs/ckpt.py",
    "src/zs/infer.py",
    "data/clean/neuesc_100.csv",
    "data/clean/vsmec_train.csv",
    "data/clean/vsmec_dev.csv",
    "data/clean/vsmec_test.csv",
    "data/subsets_phase2/MANIFEST.json",
    *[f"data/subsets_phase2/{ds}_f{f}_s{s}_ids.csv" for ds in ("neuesc", "vsmec") for f in (10, 25, 50) for s in range(5)],
]
for rel in required_repo_files:
    assert (REPO_DIR / rel).is_file(), f"Thiếu file bắt buộc tại commit {COMMIT}: {rel}"

frozen_paths = [
    p
    for folder in ("data/clean", "data/splits", "data/subsets_phase2", "src/seq", "src/zs", "src/stats", "src/phase2")
    for p in sorted((REPO_DIR / folder).rglob("*"))
    if p.is_file() and "__pycache__" not in p.parts
]
frozen_paths += [
    REPO_DIR / rel
    for rel in ("configs/base.yaml", "configs/zs.yaml", "configs/seq.yaml", "configs/phase2.yaml", "src/common/constants.py")
]
frozen_snapshot = {p.relative_to(REPO_DIR).as_posix(): sha256_path(p) for p in frozen_paths}
frozen_file = ENV_DIR / "frozen_before.json"
if frozen_file.is_file():
    assert json.loads(frozen_file.read_text(encoding="utf-8")) == frozen_snapshot, "Tài nguyên đóng băng đã thay đổi!"
else:
    frozen_file.write_text(json.dumps(frozen_snapshot, sort_keys=True, indent=2) + "\n", encoding="utf-8")

# Stage nguyên byte selection_p1_lr.json đã khóa từ Task 07
repo_selection = REPO_DIR / "results" / "phase2" / "selection_p1_lr.json"
candidate_selections = [repo_selection]
if DRIVE_SELECTION_JSON_PATH.strip():
    candidate_selections.append(Path(DRIVE_SELECTION_JSON_PATH.strip()))
if MOUNT_DRIVE and DRIVE_TASK08_DIR.is_dir():
    candidate_selections.append(DRIVE_TASK08_DIR / "selection_p1_lr.json")
if DRIVE_SESSION_ROOT is not None:
    candidate_selections.append(DRIVE_SESSION_ROOT / "selection_p1_lr.json")

chosen_selection = next((p for p in candidate_selections if p.is_file()), None)
if chosen_selection is None:
    raise FileNotFoundError(
        "Không tìm thấy selection_p1_lr.json ở bất kỳ nguồn nào: " + ", ".join(str(p) for p in candidate_selections)
    )
if SELECTION_STAGE_PATH.is_file():
    assert sha256_path(SELECTION_STAGE_PATH) == sha256_path(chosen_selection), (
        f"SELECTION_STAGE_PATH ({SELECTION_STAGE_PATH}) khác nguồn mới ({chosen_selection})!"
    )
else:
    atomic_copy_file(chosen_selection, SELECTION_STAGE_PATH)
atomic_copy_file(SELECTION_STAGE_PATH, RESULTS_DIR / "selection_p1_lr.json")
if DRIVE_SESSION_ROOT is not None:
    atomic_copy_file(SELECTION_STAGE_PATH, DRIVE_SESSION_ROOT / "selection_p1_lr.json")

if DRIVE_SESSION_ROOT is not None:
    hydrate_from_flat_drive(DRIVE_SESSION_ROOT, RESULTS_DIR, CKPT_OUT_DIR, COMMIT, PROFILE_ID, FRACTIONS_TO_RUN)

print("Commit:", COMMIT)
print("Output dir:", OUTPUT_DIR)
print("Selection JSON:", chosen_selection, "-> staged:", SELECTION_STAGE_PATH)
print("Drive backup dir:", DRIVE_SESSION_ROOT)


## 4. Kiểm tra hỗ trợ `venv`/`ensurepip` và dọn sạch venv tạo dở (nếu có)
Kiểm tra và cài đúng `pythonX.Y-venv` khi thiếu để đồng bộ Python hệ thống trước khi tạo virtualenv (`python == 3.13.18`), đồng thời xóa `VENV_DIR` tạo dở nếu thiếu `bin/python` hoặc `bin/pip`.


In [ ]:
probe = subprocess.run([sys.executable, "-c", "import venv, ensurepip"], capture_output=True, text=True)
if probe.returncode != 0:
    py_mm = f"{sys.version_info.major}.{sys.version_info.minor}"
    apt_pkg = f"python{py_mm}-venv"
    print(f"Thiếu venv/ensurepip cho Python {py_mm}; đang cài gói hệ thống {apt_pkg}...")
    apt_upd = subprocess.run(["apt-get", "update", "-qq"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    apt_ins = subprocess.run(
        ["apt-get", "install", "-y", "-qq", apt_pkg], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    apt_log = (apt_upd.stdout or "") + "\n" + (apt_ins.stdout or "")
    (LOG_DIR / f"apt_venv_{SESSION_ID}.log").write_text(apt_log, encoding="utf-8")
    print(apt_log[-3000:])
    apt_upd.check_returncode()
    apt_ins.check_returncode()
    subprocess.run([sys.executable, "-c", "import venv, ensurepip"], check=True)

sys_py_ver = subprocess.check_output([sys.executable, "-c", "import platform; print(platform.python_version())"], text=True).strip()
print(f"Host sys.executable Python: {sys_py_ver} (profile expects venv Python: {PROFILES[PROFILE_ID]['python']})")

PYTHON = VENV_DIR / "bin" / "python"
PIP_BIN = VENV_DIR / "bin" / "pip"
if VENV_DIR.exists() and not (PYTHON.is_file() and PIP_BIN.is_file()):
    print("Phát hiện virtualenv tạo dở (thiếu python hoặc pip), đang xóa để tạo lại:", VENV_DIR)
    shutil.rmtree(VENV_DIR)
print("Venv support check: PASS")


## 5. Tạo virtualenv theo profile, cài 12 package pin & cấu hình `RUN_ENV`
Kế thừa PyTorch CUDA của Colab qua `--system-site-packages`, cài đúng 12 package pin bằng `PYTHON -m pip`, ghi `install.log`, `pip-freeze.txt`, `pip_check.txt` (`check=False`), và thiết lập `RUN_ENV` (`USE_TF="0"`, `USE_TORCH="1"`, `TF_CPP_MIN_LOG_LEVEL="3"`).


In [ ]:
if not PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)], check=True)
assert PYTHON.is_file() and PIP_BIN.is_file(), f"Tạo virtualenv thất bại: {VENV_DIR}"

pins_list = [f"{name}=={ver}" for name, ver in PROFILES[PROFILE_ID]["pins"].items()]
install = subprocess.run(
    [str(PYTHON), "-m", "pip", "install", *pins_list],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
for install_path in (LOG_DIR / f"install_{SESSION_ID}.log", LOG_DIR / "install.log"):
    install_path.write_text(install.stdout, encoding="utf-8")
print(install.stdout[-4500:])
install.check_returncode()

freeze_out = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
for freeze_path in (ENV_DIR / f"pip-freeze_{SESSION_ID}.txt", ENV_DIR / "pip-freeze.txt"):
    freeze_path.write_text(freeze_out, encoding="utf-8")

pip_chk = subprocess.run(
    [str(PYTHON), "-m", "pip", "check"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False
)
pip_chk_text = (pip_chk.stdout or "") + f"\n[pip_check_exit_code]={pip_chk.returncode}\n"
for chk_path in (ENV_DIR / f"pip_check_{SESSION_ID}.txt", ENV_DIR / "pip_check.txt"):
    chk_path.write_text(pip_chk_text, encoding="utf-8")

RUN_ENV = os.environ.copy()
RUN_ENV.update(
    PYTHONUNBUFFERED="1",
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_TELEMETRY="1",
    USE_TF="0",
    USE_TORCH="1",
    TF_CPP_MIN_LOG_LEVEL="3",
)
RUN_ENV.pop("HF_HUB_OFFLINE", None)
RUN_ENV.pop("TRANSFORMERS_OFFLINE", None)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        RUN_ENV["HF_TOKEN"] = hf_token.strip()
        print("Loaded optional HF_TOKEN from Colab Secrets (redacted).")
except Exception:
    pass

def run_python(source: str) -> None:
    result = subprocess.run(
        [str(PYTHON), "-u", "-c", source],
        cwd=REPO_DIR,
        env=RUN_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    result.check_returncode()

print(f"Installed {len(pins_list)} pinned packages into {VENV_DIR} (pip check exit={pip_chk.returncode}).")


## 6. Preflight môi trường & kiểm tra forward/loss BF16 cho `DFT` bằng Python của venv
Chạy trong subprocess `PYTHON` riêng (giải phóng sạch VRAM ngay khi kết thúc): ghi `environment_<SESSION_ID>.json` **trước khi `assert`**, kiểm tra `Python == 3.13.18` + `Torch` + `CUDA` + 12 pin và chạy thử 1 bước forward/loss BF16 hữu hạn của `DFT` (`uitnlp/visobert`).


In [ ]:
preflight_env_script = r'''
import importlib.metadata as metadata
import json, platform, subprocess, sys
from pathlib import Path
import numpy, pandas, scipy, sklearn, torch, transformers, yaml
from src.phase2.models import build_phase2_model
from src.phase2.protocol import load_phase2_config
from src.phase2.run import atomic_json

ctx = json.loads(__CTX_JSON__)
profile_spec = ctx["profile_spec"]
env_dir = Path(ctx["env_dir"])

actual_pins = {}
mismatches = []
for pkg, exp_ver in profile_spec["pins"].items():
    try:
        got_ver = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        got_ver = None
    actual_pins[pkg] = got_ver
    if got_ver != exp_ver:
        mismatches.append(f"{pkg}: expected {exp_ver}, got {got_ver}")

py_ver = platform.python_version()
if py_ver != profile_spec["python"]:
    mismatches.append(f"python: expected {profile_spec['python']}, got {py_ver}")
if torch.__version__ != profile_spec["torch"]:
    mismatches.append(f"torch: expected {profile_spec['torch']}, got {torch.__version__}")
if torch.version.cuda != profile_spec["cuda"]:
    mismatches.append(f"cuda: expected {profile_spec['cuda']}, got {torch.version.cuda}")
if not torch.cuda.is_available():
    mismatches.append("torch.cuda.is_available() is False")
elif not torch.cuda.is_bf16_supported():
    mismatches.append("torch.cuda.is_bf16_supported() is False")

try:
    smi_out = subprocess.check_output(["nvidia-smi"], text=True)
except Exception as exc:
    smi_out = f"unavailable: {exc}"

snapshot = {
    "profile_id": ctx["profile_id"],
    "session_id": ctx["session_id"],
    "commit": ctx["commit"],
    "fractions_to_run": ctx["fractions_to_run"],
    "python": py_ver,
    "sys_executable": sys.executable,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "cuda_available": torch.cuda.is_available(),
    "bf16_supported": bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_total_bytes": int(torch.cuda.get_device_properties(0).total_memory) if torch.cuda.is_available() else 0,
    "nvidia_smi": smi_out,
    "pinned_packages_expected": profile_spec["pins"],
    "pinned_packages_actual": actual_pins,
    "module_files": {
        "torch": getattr(torch, "__file__", None),
        "transformers": getattr(transformers, "__file__", None),
        "numpy": getattr(numpy, "__file__", None),
        "pandas": getattr(pandas, "__file__", None),
        "sklearn": getattr(sklearn, "__file__", None),
        "scipy": getattr(scipy, "__file__", None),
        "yaml": getattr(yaml, "__file__", None),
    },
    "preflight_mismatches": mismatches,
    "preflight_status": "FAIL" if mismatches else "ENV_PASS",
}
session_env_file = env_dir / f"environment_{ctx['session_id']}.json"
atomic_json(session_env_file, snapshot)
atomic_json(env_dir / "environment.json", snapshot)

if mismatches:
    raise AssertionError(
        "Profile preflight FAILED (đã lưu chi tiết tại " + str(session_env_file) + "): " + "; ".join(mismatches)
    )

cfg = load_phase2_config("configs/phase2.yaml")
model, tok, dft_prov = build_phase2_model("dft", "SA", "ViSoBERT", 0, 25, cfg)
model = model.to("cuda").train()
enc = tok(["kiểm tra tiền huấn luyện một", "kiểm tra tiền huấn luyện hai"], padding=True, truncation=True, max_length=32, return_tensors="pt")
batch = {k: v.to("cuda") for k, v in enc.items()}
batch["labels"] = torch.tensor([0, 2], dtype=torch.long, device="cuda")
with torch.autocast("cuda", dtype=torch.bfloat16):
    out = model(**batch)
assert out.logits.shape == (2, 3) and bool(torch.isfinite(out.loss)), "DFT BF16 forward/loss preflight failed!"
del out, batch, enc, model, tok
torch.cuda.empty_cache()

snapshot["dft_pretrained_preflight"] = {"status": "PASS", "provenance": dft_prov}
atomic_json(session_env_file, snapshot)
atomic_json(env_dir / "environment.json", snapshot)
print("GPU:", snapshot["gpu"], "| Venv Python:", py_ver)
print("Environment & DFT BF16 preflight: PASS")
'''

ctx_payload = json.dumps({
    "profile_id": PROFILE_ID,
    "profile_spec": PROFILES[PROFILE_ID],
    "session_id": SESSION_ID,
    "commit": COMMIT,
    "fractions_to_run": list(FRACTIONS_TO_RUN),
    "env_dir": str(ENV_DIR),
})
try:
    run_python(preflight_env_script.replace("__CTX_JSON__", repr(ctx_payload)))
except Exception:
    if DRIVE_SESSION_ROOT is not None:
        for folder_name in ("environment", "logs"):
            for item in sorted((OUTPUT_DIR / folder_name).rglob("*")):
                if item.is_file() and item.suffix != ".tmp":
                    atomic_copy_file(item, DRIVE_SESSION_ROOT / folder_name / item.relative_to(OUTPUT_DIR / folder_name))
    raise


## 7. Stage 10 checkpoint nguồn (`seed 0..4`), xác thực `selection_p1_lr.json`, kiểm tra tập con lồng nhau `10% ⊂ 25% ⊂ 50%` & thử `Seq` BF16
Copy thật (`shutil.copytree`) cả 10 checkpoint nguồn `ViSoBERT` (`vsmec` `s0..4` và `neu3` `s0..4`) về `/content/phase2_source_ckpts`, xác minh `selection_p1_lr.json`, kiểm tra toàn bộ 20 bộ dữ liệu `main` (`{25, 50} × {SA, SB} × s{0..4}`) + **bất biến lồng nhau `10% ⊂ 25% ⊂ 50%`** trên từng seed, và thử 1 bước forward/loss BF16 cho `Seq SA` + `Seq SB`.


In [ ]:
stage_and_data_script = r'''
import json, shutil, time
from pathlib import Path
import torch
from src.common.hashing import sha256_file
from src.phase2.data import load_phase2_prepared_data
from src.phase2.models import build_phase2_model, file_hashes
from src.phase2.protocol import canonical_digest, load_phase2_config
from src.phase2.run import atomic_json
from src.phase2.subsets import load_phase2_target_splits
from src.plm.model_ckpt import _find_encoder_weight_file
from src.zs.ckpt import locate_checkpoint

ctx = json.loads(__CTX_JSON__)
stage_root = Path(ctx["stage_root"])
assert "sweep" not in str(stage_root).lower() and "smoke" not in str(stage_root).lower()
stage_root.mkdir(parents=True, exist_ok=True)

IGNORE_NAMES = {".ipynb_checkpoints", ".DS_Store", "__pycache__", "Thumbs.db"}

def ignore_junk(_dir, names):
    return {n for n in names if n in IGNORE_NAMES or n.endswith(".tmp")}

def find_matching_external_manifests(checkpoint: Path, search_root: Path, run_name: str, alias: str, domain: str):
    filename = "stage1_ckpt_hash.json" if domain == "vsmec" else "neu3_ckpt_hash.json"
    dirs = {search_root, search_root.parent, checkpoint, checkpoint.parent, checkpoint.parent.parent}
    found = []
    for d in sorted(dirs, key=str):
        for candidate in (d / filename, d / "results" / filename):
            if not candidate.is_file():
                continue
            payload = json.loads(candidate.read_text(encoding="utf-8"))
            if not isinstance(payload, dict):
                continue
            for key, entry in payload.items():
                if not isinstance(entry, dict):
                    continue
                parts = str(entry.get("path", "")).replace("\\", "/").split("/")
                if key == run_name or run_name in parts:
                    found.append(("root", filename, key, entry))
                elif (key == alias and run_name in candidate.parent.parts) or (
                    key == alias and Path(str(entry.get("path", ""))).resolve() == checkpoint.resolve()
                ):
                    found.append(("run", filename, key, entry))
    return found

staged_summary = {}
root_manifests_to_write = {}

for domain in ("vsmec", "neu3"):
    src_root = Path(ctx["search_roots"][domain])
    for seed in range(5):
        src_ckpt = locate_checkpoint(src_root, "ViSoBERT", domain, seed)
        run_folder = src_ckpt.parent if src_ckpt.name == "ViSoBERT" else src_ckpt
        run_name = run_folder.name
        dst_run_folder = stage_root / run_name

        src_enc_sha = sha256_file(_find_encoder_weight_file(src_ckpt / "encoder"))
        src_head_sha = sha256_file(src_ckpt / "head.pt")

        need_copy = True
        if dst_run_folder.is_dir():
            dst_ckpt_guess = (dst_run_folder / "ViSoBERT") if src_ckpt.name == "ViSoBERT" else dst_run_folder
            if (dst_ckpt_guess / "head.pt").is_file() and (dst_ckpt_guess / "encoder" / "config.json").is_file():
                try:
                    if (
                        sha256_file(_find_encoder_weight_file(dst_ckpt_guess / "encoder")) == src_enc_sha
                        and sha256_file(dst_ckpt_guess / "head.pt") == src_head_sha
                    ):
                        need_copy = False
                except Exception:
                    need_copy = True
            if need_copy:
                shutil.rmtree(dst_run_folder)

        if need_copy:
            shutil.copytree(run_folder, dst_run_folder, symlinks=False, ignore=ignore_junk)

        for scope, filename, key, entry in find_matching_external_manifests(src_ckpt, src_root, run_name, "ViSoBERT", domain):
            if scope == "run":
                target_manifest = dst_run_folder / "results" / filename
                if not target_manifest.is_file() and not (dst_run_folder / filename).is_file():
                    atomic_json(target_manifest, {key: entry})
            else:
                root_manifests_to_write.setdefault(filename, {})[key] = entry

for filename, payload in root_manifests_to_write.items():
    target_manifest = stage_root / filename
    merged = json.loads(target_manifest.read_text(encoding="utf-8")) if target_manifest.is_file() else {}
    merged.update(payload)
    atomic_json(target_manifest, merged)

TASK07_S0_ENC_SHA = {
    "vsmec": "3cf5ebc839dca08f73e654a0f5090309c9aa1f515e6e20a9b3e29009a5300805",
    "neu3": "da379459f87d4404b09ee206c5a426f814d47f7f696f00b6a7a92e41affb3bb4",
}

for domain in ("vsmec", "neu3"):
    src_root = Path(ctx["search_roots"][domain])
    for seed in range(5):
        src_ckpt = locate_checkpoint(src_root, "ViSoBERT", domain, seed)
        staged_ckpt = locate_checkpoint(stage_root, "ViSoBERT", domain, seed)
        assert not staged_ckpt.is_symlink(), f"Staged checkpoint không được là symlink: {staged_ckpt}"
        enc_sha = sha256_file(_find_encoder_weight_file(staged_ckpt / "encoder"))
        head_sha = sha256_file(staged_ckpt / "head.pt")
        assert enc_sha == sha256_file(_find_encoder_weight_file(src_ckpt / "encoder")), f"Encoder SHA mismatch ({domain} s{seed})"
        assert head_sha == sha256_file(src_ckpt / "head.pt"), f"Head SHA mismatch ({domain} s{seed})"
        if ctx.get("verify_task07_s0_sha", True) and seed == 0:
            assert enc_sha == TASK07_S0_ENC_SHA[domain], (
                f"Checkpoint {domain} seed=0 không khớp Task 07-08: {enc_sha} != {TASK07_S0_ENC_SHA[domain]}"
            )
        staged_summary[f"{domain}_s{seed}"] = {
            "domain": domain,
            "seed": seed,
            "drive_source": str(src_ckpt),
            "staged_path": str(staged_ckpt),
            "encoder_sha256": enc_sha,
            "head_sha256": head_sha,
            "n_files": len(file_hashes(staged_ckpt)),
        }
        print(f"Staged & verified ({domain} s{seed}): {staged_ckpt.name} | enc_sha={enc_sha[:12]}...")

cfg = load_phase2_config("configs/phase2.yaml")
assert cfg["protocol_version"] == "phase2-v1"
assert cfg["training"]["strict_precision"] is True and cfg["training"]["precision"] == "bf16"

sel_path = Path(ctx["selection_path"])
sel = json.loads(sel_path.read_text(encoding="utf-8"))
assert sel.get("protocol_version") == "phase2-v1", "selection_p1_lr.json sai protocol_version"
assert sel.get("protocol_hash") == canonical_digest(cfg), "selection_p1_lr.json sai protocol_hash"
expected_lrs = {
    "ViSoBERT/SA/dft": 3.0e-5,
    "ViSoBERT/SA/seq": 2.0e-5,
    "ViSoBERT/SB/dft": 2.0e-5,
    "ViSoBERT/SB/seq": 3.0e-5,
}
for k, exp_lr in expected_lrs.items():
    got_lr = float(sel["selected"][k]["lr"])
    assert abs(got_lr - exp_lr) < 1e-12, f"LR khóa cho {k} là {got_lr}, mong đợi {exp_lr}"
print("Locked selection_p1_lr.json verified:", {k: float(sel["selected"][k]["lr"]) for k in expected_lrs})

expected_main_shapes = {
    ("SA", 25): (5762, 3305, 6613, 3),
    ("SA", 50): (11524, 3305, 6613, 3),
    ("SB", 25): (1386, 686, 692, 7),
    ("SB", 50): (2771, 686, 692, 7),
}
data_prov = {}
seq_prov = {}
fractions_to_run = [int(f) for f in ctx["fractions_to_run"]]

for direction, domain, num_labels in (("SA", "neuesc", 3), ("SB", "vsmec", 7)):
    for seed in range(5):
        tr10, _, _, _ = load_phase2_target_splits(domain, 10, seed, include_test=False)
        ids_10 = set(tr10["id"].astype(int).tolist())
        ids_by_frac = {10: ids_10}
        for frac in fractions_to_run:
            exp_tr, exp_dv, exp_te, exp_nl = expected_main_shapes[(direction, frac)]
            assert exp_nl == num_labels
            prepared = load_phase2_prepared_data(direction, frac, seed, "ViSoBERT", cfg, run_mode="main")
            assert len(prepared["train_df"]) == exp_tr, (direction, frac, seed)
            assert len(prepared["dev_df"]) == exp_dv, (direction, frac, seed)
            assert len(prepared["test_df"]) == exp_te, (direction, frac, seed)
            assert prepared["provenance"]["test_integrity_checked"] is True, (direction, frac, seed)
            src_data_digest = canonical_digest(prepared["provenance"]["source_files_sha256"])
            assert src_data_digest == sel["selected"][f"ViSoBERT/{direction}/dft"]["data_hash"]
            assert src_data_digest == sel["selected"][f"ViSoBERT/{direction}/seq"]["data_hash"]
            data_prov[f"{direction}_f{frac}_s{seed}"] = prepared["provenance"]
            ids_by_frac[frac] = set(prepared["train_df"]["id"].astype(int).tolist())

            if seed == 0 and frac == fractions_to_run[0]:
                model, tok, prov = build_phase2_model(
                    "seq", direction, "ViSoBERT", 0, frac, cfg, source_ckpt_root=stage_root
                )
                model = model.to("cuda").train()
                enc = tok(prepared["train_texts"][:2], padding=True, truncation=True, max_length=cfg["training"]["max_length"], return_tensors="pt")
                batch = {k: v.to("cuda") for k, v in enc.items()}
                batch["labels"] = torch.tensor(prepared["train_labels"][:2], dtype=torch.long, device="cuda")
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out = model(**batch)
                assert out.logits.shape == (2, num_labels) and bool(torch.isfinite(out.loss)), f"Seq {direction} f{frac} BF16 preflight failed!"
                del out, batch, enc, model, tok
                torch.cuda.empty_cache()
                seq_prov[direction] = prov

        if 25 in ids_by_frac:
            assert ids_10 < ids_by_frac[25], f"Vi phạm lồng nhau 10% < 25% tại {direction} s{seed}"
        if 50 in ids_by_frac:
            prev_f = 25 if 25 in ids_by_frac else 10
            assert ids_by_frac[prev_f] < ids_by_frac[50], f"Vi phạm lồng nhau {prev_f}% < 50% tại {direction} s{seed}"
    print(f"Nested subsets (10% < 25% < 50%, s0..4) + Seq BF16 preflight ({direction}): PASS")

env_dir = Path(ctx["env_dir"])
session_env_file = env_dir / f"environment_{ctx['session_id']}.json"
snapshot = json.loads(session_env_file.read_text(encoding="utf-8"))
snapshot.update({
    "preflight_status": "PASS",
    "setup_seconds_before_main": round(time.perf_counter() - float(ctx["t0_setup"]), 3),
    "selection_sha256": sha256_file(sel_path),
    "data_provenance": data_prov,
    "source_ckpt_provenance_s0": seq_prov,
    "staged_source_checkpoints": staged_summary,
})
atomic_json(session_env_file, snapshot)
atomic_json(env_dir / "environment.json", snapshot)
print("Full Step 7 preflight & 10-checkpoint + nested subset recording: PASS")
'''

stage_ctx = json.dumps({
    "stage_root": str(SOURCE_CKPT_STAGE_DIR),
    "search_roots": {k: str(v) for k, v in CKPT_SEARCH_ROOTS.items()},
    "selection_path": str(SELECTION_STAGE_PATH),
    "fractions_to_run": list(FRACTIONS_TO_RUN),
    "env_dir": str(ENV_DIR),
    "session_id": SESSION_ID,
    "t0_setup": T0_SETUP,
    "verify_task07_s0_sha": True,
})
run_python(stage_and_data_script.replace("__CTX_JSON__", repr(stage_ctx)))

if DRIVE_SESSION_ROOT is not None:
    for folder_name in ("environment", "logs"):
        for item in sorted((OUTPUT_DIR / folder_name).rglob("*")):
            if item.is_file() and item.suffix != ".tmp":
                atomic_copy_file(item, DRIVE_SESSION_ROOT / folder_name / item.relative_to(OUTPUT_DIR / folder_name))
    print("Synced preflight environment & install logs once to Drive:", DRIVE_SESSION_ROOT)


## 8. Thực thi tuần tự 52 lượt `main` (`25%` trước → `50%` sau) & đồng bộ lên Drive phẳng
Chạy trọn vẹn **toàn bộ 26 lượt của `25%` trước**, rồi mới chạy **26 lượt của `50%`** theo đúng `cut_order` trong `configs/phase2.yaml`. Ở mỗi mức `fraction`: `SA (5 DFT → 3 C0 → 5 Seq)` → `SB (5 DFT → 3 C0 → 5 Seq)`. Hàm `find_completed_dft_run_dir` kiểm tra nghiêm ngặt đúng **`(direction, fraction, seed)`** để `C0 50%` chỉ nạp `DFT 50%`. Sau mỗi lượt hoàn tất: xóa `work/<run_id>/resume.pt` (giữ `provenance.json`) và đồng bộ nguyên tử lên Drive.


In [ ]:
selection_payload = json.loads(SELECTION_STAGE_PATH.read_text(encoding="utf-8"))
selected_map = selection_payload["selected"]

MAIN_JOBS = []
for fraction in FRACTIONS_TO_RUN:
    for direction in ("SA", "SB"):
        dft_lr = float(selected_map[f"ViSoBERT/{direction}/dft"]["lr"])
        seq_lr = float(selected_map[f"ViSoBERT/{direction}/seq"]["lr"])
        for seed in range(5):
            MAIN_JOBS.append((fraction, direction, "dft", seed, dft_lr))
        for seed in range(3):
            MAIN_JOBS.append((fraction, direction, "c0", seed, seq_lr))
        for seed in range(5):
            MAIN_JOBS.append((fraction, direction, "seq", seed, seq_lr))

TOTAL_JOBS = len(MAIN_JOBS)
assert TOTAL_JOBS == 26 * len(FRACTIONS_TO_RUN), TOTAL_JOBS

TRAIN_LOG_PATH = LOG_DIR / f"train_{SESSION_ID}.log"
COMMANDS_RECORD = []

def stop_process_group(proc: subprocess.Popen) -> None:
    if proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGINT)
            proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(timeout=30)
        except ProcessLookupError:
            pass

def find_completed_dft_run_dir(results_dir: Path, direction: str, fraction: int, seed: int) -> Path:
    matches = []
    runs_root = results_dir / "runs"
    if not runs_root.is_dir():
        raise FileNotFoundError(f"Chưa có thư mục runs tại {runs_root}")
    for comp_file in sorted(runs_root.glob("*/completion.json")):
        rdir = comp_file.parent
        meta_file = rdir / "metadata.json"
        if not meta_file.is_file():
            continue
        comp = json.loads(comp_file.read_text(encoding="utf-8"))
        meta = json.loads(meta_file.read_text(encoding="utf-8"))
        if (
            comp.get("complete") is True
            and comp.get("ckpt_path")
            and meta.get("direction") == direction
            and meta.get("method") == "dft"
            and int(meta.get("fraction", -1)) == int(fraction)
            and int(meta.get("seed", -1)) == int(seed)
            and meta.get("run_mode") == "main"
            and meta.get("backbone") == "ViSoBERT"
        ):
            matches.append(rdir)
    assert len(matches) == 1, f"Cần đúng 1 lượt DFT hoàn tất cho ({direction}, f{fraction}, s{seed}), tìm thấy {len(matches)}: {matches}"
    return matches[0]

def should_backup_ckpt(method: str, seed: int) -> bool:
    if method != "dft" and not SAVE_ALL_MAIN_CKPTS:
        return False
    if BACKUP_DFT_CKPTS_TO_DRIVE == "all":
        return True
    if BACKUP_DFT_CKPTS_TO_DRIVE == "c0_only":
        return method == "dft" and int(seed) in (0, 1, 2)
    return False

def sync_completed_main_run_to_drive(run_id: str, direction: str, fraction: int, method: str, seed: int) -> None:
    if DRIVE_SESSION_ROOT is None:
        return
    d_res = DRIVE_SESSION_ROOT
    local_prov = RESULTS_DIR / "work" / run_id / "provenance.json"
    local_run_dir = RESULTS_DIR / "runs" / run_id
    atomic_copy_file(local_prov, d_res / "work" / run_id / "provenance.json")

    if should_backup_ckpt(method, seed):
        local_ckpt_dir = CKPT_OUT_DIR / run_id
        drive_ckpt_dir = d_res / "checkpoints" / run_id
        for item in sorted(local_ckpt_dir.rglob("*")):
            if item.is_file() and item.suffix != ".tmp":
                dst = drive_ckpt_dir / item.relative_to(local_ckpt_dir)
                if dst.is_file() and dst.stat().st_size == item.stat().st_size and sha256_path(dst) == sha256_path(item):
                    continue
                atomic_copy_file(item, dst)

    for item in sorted(local_run_dir.rglob("*")):
        if item.is_file() and item.name != "completion.json" and item.suffix != ".tmp":
            atomic_copy_file(item, d_res / "runs" / run_id / item.relative_to(local_run_dir))
    atomic_copy_file(local_run_dir / "completion.json", d_res / "runs" / run_id / "completion.json")
    atomic_copy_file(RESULTS_DIR / "phase2_runs.csv", d_res / "phase2_runs.csv")

for idx, (fraction, direction, method, seed, lr) in enumerate(MAIN_JOBS, start=1):
    cmd = [
        str(PYTHON), "-u", "-m", "src.phase2.cli",
        "--config", "configs/phase2.yaml",
        "--backbone", "ViSoBERT",
        "--direction", direction,
        "--method", method,
        "--fraction", str(fraction),
        "--seed", str(seed),
        "--lr", f"{lr:g}",
        "--run-mode", "main",
        "--selection-path", str(SELECTION_STAGE_PATH),
        "--results-dir", str(RESULTS_DIR),
        "--ckpt-out-dir", str(CKPT_OUT_DIR),
    ]
    if method == "seq":
        cmd += ["--source-ckpt-root", str(SOURCE_CKPT_STAGE_DIR)]
    elif method == "c0":
        dft_run_dir = find_completed_dft_run_dir(RESULTS_DIR, direction, fraction, seed)
        cmd += ["--c0-stage1-run-dir", str(dft_run_dir)]

    if method != "dft" and not SAVE_ALL_MAIN_CKPTS:
        cmd.append("--no-save-ckpt")

    COMMANDS_RECORD.append({
        "index": idx, "fraction": fraction, "direction": direction, "method": method, "seed": seed, "lr": lr, "argv": cmd
    })
    (LOG_DIR / f"commands_{SESSION_ID}.json").write_text(
        json.dumps(COMMANDS_RECORD, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )

    with TRAIN_LOG_PATH.open("a", encoding="utf-8") as log_handle:
        start_offset = log_handle.tell()
        header = (
            f"\n=== [{idx}/{TOTAL_JOBS}] f{fraction}/{direction}/{method}/s{seed}/lr={lr:g} start "
            f"{time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())} ===\n"
        )
        log_handle.write(header)
        log_handle.flush()

        t_job = time.perf_counter()
        proc = subprocess.Popen(
            cmd,
            cwd=REPO_DIR,
            env=RUN_ENV,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        last_heartbeat = time.monotonic()
        try:
            while proc.poll() is None:
                time.sleep(3)
                if time.monotonic() - last_heartbeat >= 30:
                    last_heartbeat = time.monotonic()
                    tail_txt = TRAIN_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-500:]
                    last_line = [ln for ln in tail_txt.splitlines() if ln.strip()][-1:] or ["..."]
                    print(f"  [{idx:02d}/{TOTAL_JOBS} f{fraction} {direction}/{method}/s{seed} {round(time.perf_counter() - t_job)}s] {last_line[0]}")
        except BaseException:
            stop_process_group(proc)
            if DRIVE_SESSION_ROOT is not None and TRAIN_LOG_PATH.is_file():
                atomic_copy_file(TRAIN_LOG_PATH, DRIVE_SESSION_ROOT / "logs" / TRAIN_LOG_PATH.name)
            raise

    with TRAIN_LOG_PATH.open("r", encoding="utf-8", errors="replace") as read_handle:
        read_handle.seek(start_offset)
        segment = read_handle.read()

    if proc.returncode != 0:
        if DRIVE_SESSION_ROOT is not None and TRAIN_LOG_PATH.is_file():
            atomic_copy_file(TRAIN_LOG_PATH, DRIVE_SESSION_ROOT / "logs" / TRAIN_LOG_PATH.name)
        print(segment[-6000:])
        raise RuntimeError(f"Lượt [{idx}/{TOTAL_JOBS}] (f{fraction}/{direction}/{method}/s{seed}) thất bại (exit={proc.returncode}).")

    matches = re.findall(r"^([A-Za-z0-9._\-]+): (complete|SKIP)\s*$", segment, flags=re.MULTILINE)
    assert matches, f"Không tìm thấy '<run_id>: complete|SKIP' ở lượt [{idx}/{TOTAL_JOBS}].\n{segment[-1500:]}"
    run_id, run_status = matches[-1]

    comp_path = RESULTS_DIR / "runs" / run_id / "completion.json"
    prov_path = RESULTS_DIR / "work" / run_id / "provenance.json"
    assert comp_path.is_file() and json.loads(comp_path.read_text(encoding="utf-8")).get("complete") is True
    assert prov_path.is_file(), f"Thiếu provenance.json cho {run_id}!"

    if CLEAN_COMPLETED_RESUME_PT:
        resume_pt = RESULTS_DIR / "work" / run_id / "resume.pt"
        if resume_pt.is_file():
            resume_pt.unlink()
        assert prov_path.is_file()

    sync_completed_main_run_to_drive(run_id, direction, fraction, method, seed)

    metrics_obj = json.loads((RESULTS_DIR / "runs" / run_id / "metrics.json").read_text(encoding="utf-8"))
    cost_obj = json.loads((RESULTS_DIR / "runs" / run_id / "cost.json").read_text(encoding="utf-8"))
    primary_key = "V3-NEU/none" if direction == "SA" else "V7-VSMEC/none"
    test_f1 = metrics_obj["test_metrics"][primary_key]["macro_f1"]
    print(
        f"[{idx:02d}/{TOTAL_JOBS}] {run_id} | {run_status} | "
        f"dev={metrics_obj['dev_selection_score']:.4f} | test({primary_key.split('/')[0]})={test_f1:.4f} | "
        f"best_ep={metrics_obj['best_epoch']} | steps={cost_obj['actual_update_steps']} | "
        f"train_s={cost_obj['train_seconds']:.1f}s"
    )

print(f"Completed all {TOTAL_JOBS} Task09 main runs (fractions={FRACTIONS_TO_RUN}).")


## 9. Kiểm tra `SKIP` và nghiệm thu độc lập toàn bộ 52 lượt `main` (`25%` + `50%`)
Gọi lại CLI trả `SKIP` cho 12 cấu hình đại diện (`{25, 50} × {SA, SB} × {dft, c0, seq}`, `seed=0`), rồi kiểm định độc lập toàn bộ `52` `completion.json`, `52` `provenance.json`, `20` checkpoint `DFT`, `12` lượt `C0` đúng `(direction, fraction, seed)`, `52` bộ `predictions_dev.csv` & `predictions_test.csv`, **`52` `run_id` / `260` dòng (`78` `dev` + `182` `test`)** trong `phase2_runs.csv` và tính toàn vẹn `frozen_before.json`.


In [ ]:
selection_payload = json.loads(SELECTION_STAGE_PATH.read_text(encoding="utf-8"))
selected_map = selection_payload["selected"]
skip_logs = []

for fraction in FRACTIONS_TO_RUN:
    for direction in ("SA", "SB"):
        dft_lr = float(selected_map[f"ViSoBERT/{direction}/dft"]["lr"])
        seq_lr = float(selected_map[f"ViSoBERT/{direction}/seq"]["lr"])
        dft_s0_dir = find_completed_dft_run_dir(RESULTS_DIR, direction, fraction, 0)
        for method, lr, c0_dir in (("dft", dft_lr, None), ("c0", seq_lr, dft_s0_dir), ("seq", seq_lr, None)):
            cmd = [
                str(PYTHON), "-u", "-m", "src.phase2.cli",
                "--config", "configs/phase2.yaml",
                "--backbone", "ViSoBERT",
                "--direction", direction,
                "--method", method,
                "--fraction", str(fraction),
                "--seed", "0",
                "--lr", f"{lr:g}",
                "--run-mode", "main",
                "--selection-path", str(SELECTION_STAGE_PATH),
                "--results-dir", str(RESULTS_DIR),
                "--ckpt-out-dir", str(CKPT_OUT_DIR),
            ]
            if method == "seq":
                cmd += ["--source-ckpt-root", str(SOURCE_CKPT_STAGE_DIR)]
            elif method == "c0":
                cmd += ["--c0-stage1-run-dir", str(c0_dir)]
            if method != "dft" and not SAVE_ALL_MAIN_CKPTS:
                cmd.append("--no-save-ckpt")

            res = subprocess.run(cmd, cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            skip_logs.append(f"$ {' '.join(cmd)}\n{res.stdout}")
            print(res.stdout.strip())
            res.check_returncode()
            assert ": SKIP" in res.stdout, f"Lượt f{fraction}/{direction}/{method}/s0 không SKIP!"

(LOG_DIR / "skip_check.log").write_text("\n".join(skip_logs) + "\n", encoding="utf-8")

verify_main_script = r'''
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
from src.common.hashing import sha256_file
from src.common.schema import ZS_PRED_COLUMNS
from src.phase2.protocol import canonical_digest, compute_dev_selection_score, load_phase2_config
from src.phase2.run import required_views, score_view
from src.plm.model_ckpt import _find_encoder_weight_file

ctx = json.loads(__CTX_JSON__)
results_dir = Path(ctx["results_dir"])
ckpt_out_dir = Path(ctx["ckpt_out_dir"])
env_dir = Path(ctx["env_dir"])
commit = ctx["commit"]
fractions_run = [int(f) for f in ctx["fractions_to_run"]]
n_fracs = len(fractions_run)
n_expected_runs = 26 * n_fracs
save_all_main_ckpts = bool(ctx["save_all_main_ckpts"])
eval_shapes = ctx.get("expected_eval_shapes", {"SA": [3305, 6613], "SB": [686, 692, 526]})
sel = json.loads(Path(ctx["selection_path"]).read_text(encoding="utf-8"))

cfg = load_phase2_config("configs/phase2.yaml")
expected_lrs = {
    ("SA", "dft"): float(sel["selected"]["ViSoBERT/SA/dft"]["lr"]),
    ("SA", "c0"): float(sel["selected"]["ViSoBERT/SA/seq"]["lr"]),
    ("SA", "seq"): float(sel["selected"]["ViSoBERT/SA/seq"]["lr"]),
    ("SB", "dft"): float(sel["selected"]["ViSoBERT/SB/dft"]["lr"]),
    ("SB", "c0"): float(sel["selected"]["ViSoBERT/SB/seq"]["lr"]),
    ("SB", "seq"): float(sel["selected"]["ViSoBERT/SB/seq"]["lr"]),
}

manifests = sorted((results_dir / "runs").glob("*/completion.json"))
assert len(manifests) == n_expected_runs, f"Cần {n_expected_runs} completion.json, tìm thấy {len(manifests)}"
prov_files = sorted((results_dir / "work").glob("*/provenance.json"))
assert len(prov_files) == n_expected_runs, f"Cần {n_expected_runs} provenance.json, tìm thấy {len(prov_files)}"

code_digests = set()
seen_combos = set()
dft_meta_map = {}
c0_records = []
test_ids_by_dir = {}
required_artifacts = {"config.yaml", "metadata.json", "metrics.json", "predictions_dev.csv", "predictions_test.csv", "cost.json"}

for manifest_path in manifests:
    run_dir = manifest_path.parent
    run_id = run_dir.name
    comp = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert comp.get("complete") is True, run_id
    artifacts = comp.get("artifacts", {})
    assert required_artifacts.issubset(artifacts), run_id
    for rel, digest in artifacts.items():
        target = (run_dir / rel).resolve()
        assert target.is_file() and sha256_file(target) == digest, f"Hash lệch tại {run_id}/{rel}"

    prov_path = results_dir / "work" / run_id / "provenance.json"
    prov = json.loads(prov_path.read_text(encoding="utf-8"))
    assert prov.get("fingerprint") == comp.get("fingerprint"), f"Fingerprint lệch cho {run_id}"
    code_digests.add(canonical_digest(prov["source_code_sha256"]))

    meta = json.loads((run_dir / "metadata.json").read_text(encoding="utf-8"))
    assert meta["run_id"] == run_id and meta["run_mode"] == "main", run_id
    assert meta["backbone"] == "ViSoBERT" and int(meta["fraction"]) in set(fractions_run), run_id
    assert meta["max_examples"] is None and meta["git_commit"] == commit, run_id
    frac, direction, method, seed = int(meta["fraction"]), meta["direction"], meta["method"], int(meta["seed"])
    assert abs(float(meta["lr"]) - expected_lrs[(direction, method)]) < 1e-12, run_id
    seen_combos.add((frac, direction, method, seed))

    if method == "dft":
        assert comp.get("ckpt_path"), f"DFT thiếu ckpt_path: {run_id}"
        ckpt_resolved = (run_dir / comp["ckpt_path"]).resolve()
        assert ckpt_resolved == (ckpt_out_dir / run_id).resolve(), run_id
        assert (ckpt_resolved / "head.pt").is_file() and _find_encoder_weight_file(ckpt_resolved / "encoder").is_file()
        dft_meta_map[(frac, direction, seed)] = meta
    elif not save_all_main_ckpts:
        assert comp.get("ckpt_path") is None, f"{run_id} lẽ ra không lưu checkpoint main"

    if method == "c0":
        c0_records.append((frac, direction, seed, meta, json.loads((run_dir / "cost.json").read_text(encoding="utf-8"))))

    metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    cost = json.loads((run_dir / "cost.json").read_text(encoding="utf-8"))
    assert isinstance(metrics["test_metrics"], dict) and len(metrics["history"]) >= 1, run_id
    assert all(math.isfinite(h["train_loss"]) for h in metrics["history"]), run_id
    assert cost["effective_precision"] == "bf16" and cost["train_seconds"] > 0, run_id
    assert cost["total_params"] == cost["trainable_params"] > 0, run_id

    for split in ("dev", "test"):
        pred = pd.read_csv(run_dir / f"predictions_{split}.csv")
        assert tuple(pred.columns) == ZS_PRED_COLUMNS and pred["id"].is_unique, (run_id, split)
        p3 = pred[["p3_neg", "p3_neu", "p3_pos"]].to_numpy(float)
        assert np.isfinite(p3).all() and (p3 >= 0).all() and np.allclose(p3.sum(axis=1), 1.0, atol=1e-6), (run_id, split)
        if direction == "SA":
            assert len(pred) == (eval_shapes["SA"][0] if split == "dev" else eval_shapes["SA"][1]), (run_id, split)
        else:
            assert len(pred) == (eval_shapes["SB"][0] if split == "dev" else eval_shapes["SB"][1]), (run_id, split)
            if split == "test" and len(eval_shapes["SB"]) >= 3:
                assert int(pred["in_strict"].astype(bool).sum()) == eval_shapes["SB"][2], run_id
            p7 = pred[[f"p7_{i}" for i in range(7)]].to_numpy(float)
            assert np.isfinite(p7).all() and (p7 >= 0).all() and np.allclose(p7.sum(axis=1), 1.0, atol=1e-6), (run_id, split)

        if split == "dev":
            recomp_dev = compute_dev_selection_score(
                pred["y_true3" if direction == "SA" else "y_true7"],
                p3.argmax(axis=1) if direction == "SA" else pred[[f"p7_{i}" for i in range(7)]].to_numpy(float).argmax(axis=1),
                "neuesc" if direction == "SA" else "vsmec",
            )
            assert np.isclose(recomp_dev, metrics["dev_selection_score"], rtol=0, atol=1e-12), run_id
        else:
            ids_tuple = tuple(pred["id"].tolist())
            if direction not in test_ids_by_dir:
                test_ids_by_dir[direction] = ids_tuple
            else:
                assert test_ids_by_dir[direction] == ids_tuple, f"Thứ tự test ID lệch giữa các lượt {direction}: {run_id}"
            for s_name, view, mapping in required_views(direction, "main"):
                if s_name != "test":
                    continue
                recomp = score_view(pred, view, mapping)
                recorded = metrics["test_metrics"][f"{view}/{mapping}"]
                assert np.isclose(recomp["macro_f1"], recorded["macro_f1"], rtol=0, atol=1e-12), (run_id, view, mapping)
                if view == "V3-VSMEC-strict":
                    assert json.loads(recorded["f1_per_class"])[1] is None, (run_id, view, mapping)

expected_combos = (
    {(f, d, "dft", s) for f in fractions_run for d in ("SA", "SB") for s in range(5)}
    | {(f, d, "c0", s) for f in fractions_run for d in ("SA", "SB") for s in range(3)}
    | {(f, d, "seq", s) for f in fractions_run for d in ("SA", "SB") for s in range(5)}
)
assert seen_combos == expected_combos, f"Tập {n_expected_runs} cấu hình không khớp: {seen_combos ^ expected_combos}"
assert len(code_digests) == 1, "source_code_sha256 không đồng nhất giữa các lượt!"

assert len(c0_records) == 6 * n_fracs
for frac, direction, seed, meta, cost in c0_records:
    dft_meta = dft_meta_map[(frac, direction, seed)]
    prov = meta["source_ckpt_provenance"]
    assert prov["stage1_dft_run_id"] == dft_meta["run_id"]
    assert int(dft_meta["fraction"]) == frac
    assert prov["stage1_ids_sha256"] == meta["data_provenance"]["ids_sha256"] == dft_meta["data_provenance"]["ids_sha256"]
    assert Path(cost["stage1_dft_cost_ref"]).resolve() == (results_dir / "runs" / dft_meta["run_id"] / "cost.json").resolve()

rows = pd.read_csv(results_dir / "phase2_runs.csv", keep_default_na=False)
assert rows["run_id"].nunique() == n_expected_runs, f"Cần {n_expected_runs} run_id duy nhất, có {rows['run_id'].nunique()}"
assert len(rows) == 130 * n_fracs, f"Cần {130 * n_fracs} dòng, có {len(rows)}"
assert int(rows["split"].eq("dev").sum()) == 39 * n_fracs and int(rows["split"].eq("test").sum()) == 91 * n_fracs
assert int((rows["direction"].eq("SA") & rows["split"].eq("test")).sum()) == 13 * n_fracs
assert int((rows["direction"].eq("SB") & rows["split"].eq("test")).sum()) == 78 * n_fracs
assert rows["run_mode"].eq("main").all() and set(rows["fraction"].astype(int).tolist()) == set(fractions_run)
assert rows["max_examples"].eq("").all() and rows["git_commit"].eq(commit).all()

frozen = json.loads((env_dir / "frozen_before.json").read_text(encoding="utf-8"))
assert all(sha256_file(Path(rel)) == digest for rel, digest in frozen.items()), "Tài nguyên đóng băng bị thay đổi!"

print(f"Independent verification of all {n_expected_runs} main runs ({130 * n_fracs} CSV rows), C0 fraction binding, test IDs, SKIP & frozen files: PASS")
'''

verify_ctx = json.dumps({
    "results_dir": str(RESULTS_DIR),
    "ckpt_out_dir": str(CKPT_OUT_DIR),
    "env_dir": str(ENV_DIR),
    "selection_path": str(SELECTION_STAGE_PATH),
    "commit": COMMIT,
    "fractions_to_run": list(FRACTIONS_TO_RUN),
    "save_all_main_ckpts": SAVE_ALL_MAIN_CKPTS,
})
run_python(verify_main_script.replace("__CTX_JSON__", repr(verify_ctx)))


## 10. Xuất 4 bảng tổng hợp đường cong `10% → 25% → 50%` (`Strict` per-class, `exploratory=True`), sinh báo cáo & đồng bộ Drive
Ghép (tùy chọn) kết quả `10%` từ `DRIVE_TASK08_DIR` với `25%` & `50%` vừa chạy để xuất 4 bảng CSV: `task09_manifest_52runs.csv`, `task09_metrics_by_fraction.csv`, `task09_per_class_f1_by_fraction.csv` (`V3-VSMEC-strict` chỉ xuất `negative` & `positive`), và `task09_contrasts_scaling.csv` (toàn bộ `25%`/`50%` mang `exploratory=True`, `p_holm=NaN`, tách bạch `point_sign` và `ci_excludes_zero`), viết `reports/phase2/task09_scaling_25_50pct.md` và đồng bộ lên `/content/drive/MyDrive/Xdomain/Phase2/Task09`.


In [ ]:
analyze_and_report_script = r'''
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from src.common.constants import EMOTIONS7, ID_TO_LABEL3
from src.phase2.protocol import apply_phase2_holm, bootstrap_seed_sd_ratio, load_phase2_config
from src.seq.analysis import paired_gain
from src.zs.infer import _atomic_csv
from src.zs.views import strict_mask

ctx = json.loads(__CTX_JSON__)
output_dir = Path(ctx["output_dir"])
results_dir = Path(ctx["results_dir"])
env_dir = Path(ctx["env_dir"])
fractions_run = sorted(int(f) for f in ctx["fractions_to_run"])
task08_dir = Path(ctx["task08_dir"]) if ctx.get("task08_dir") else None
include_task08 = bool(ctx.get("include_task08", True))

cfg = load_phase2_config("configs/phase2.yaml")
env = json.loads((env_dir / "environment.json").read_text(encoding="utf-8"))
rows_t9 = pd.read_csv(results_dir / "phase2_runs.csv", keep_default_na=False)
tables_dir = results_dir / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

def load_completed_runs_from_root(root_dir: Path, expected_fractions: set[int]):
    runs_map = {}
    preds = {}
    costs = {}
    for meta_file in sorted((root_dir / "runs").glob("*/metadata.json")):
        rdir = meta_file.parent
        comp_file = rdir / "completion.json"
        pred_test = rdir / "predictions_test.csv"
        if not (comp_file.is_file() and pred_test.is_file()):
            continue
        comp = json.loads(comp_file.read_text(encoding="utf-8"))
        if comp.get("complete") is not True:
            continue
        meta = json.loads(meta_file.read_text(encoding="utf-8"))
        frac = int(meta["fraction"])
        if meta.get("run_mode") != "main" or frac not in expected_fractions:
            continue
        metrics = json.loads((rdir / "metrics.json").read_text(encoding="utf-8"))
        cost = json.loads((rdir / "cost.json").read_text(encoding="utf-8"))
        key = (meta["direction"], frac, meta["method"], int(meta["seed"]))
        runs_map[key] = (meta, metrics, cost, comp, rdir)
        costs[meta["run_id"]] = cost
        preds[key] = pd.read_csv(pred_test)
    return runs_map, preds, costs

runs_by_key, preds_cache, cost_cache = load_completed_runs_from_root(results_dir, set(fractions_run))

# Optionally read Task 08 (10%, 26 runs) from DRIVE_TASK08_DIR for unified 10% -> 25% -> 50% curve
has_task08_10pct = False
task08_gpu = None
rows_all_list = [rows_t9]
if include_task08 and task08_dir is not None and (task08_dir / "phase2_runs.csv").is_file() and (task08_dir / "runs").is_dir():
    t8_runs, t8_preds, t8_costs = load_completed_runs_from_root(task08_dir, {10})
    expected_t8_keys = (
        {(d, 10, "dft", s) for d in ("SA", "SB") for s in range(5)}
        | {(d, 10, "c0", s) for d in ("SA", "SB") for s in range(3)}
        | {(d, 10, "seq", s) for d in ("SA", "SB") for s in range(5)}
    )
    if expected_t8_keys.issubset(t8_runs.keys()):
        # Verify test ID alignment & protocol_version == phase2-v1 before merging
        ids_ok = True
        for d in ("SA", "SB"):
            ref_t9_ids = preds_cache[(d, fractions_run[0], "dft", 0)]["id"].tolist()
            t8_meta_0 = t8_runs[(d, 10, "dft", 0)][0]
            if t8_meta_0.get("protocol_version") != "phase2-v1":
                ids_ok = False
                break
            for k_t8 in [k for k in expected_t8_keys if k[0] == d]:
                if t8_preds[k_t8]["id"].tolist() != ref_t9_ids:
                    ids_ok = False
                    break
        if ids_ok:
            has_task08_10pct = True
            runs_by_key.update(t8_runs)
            preds_cache.update(t8_preds)
            cost_cache.update(t8_costs)
            rows_t8 = pd.read_csv(task08_dir / "phase2_runs.csv", keep_default_na=False)
            rows_all_list.insert(0, rows_t8.loc[rows_t8["fraction"].astype(int).eq(10) & rows_t8["run_mode"].eq("main")])
            t8_env_file = task08_dir / "environment" / "environment.json"
            if t8_env_file.is_file():
                task08_gpu = json.loads(t8_env_file.read_text(encoding="utf-8")).get("gpu")

rows_all_df = pd.concat(rows_all_list, ignore_index=True)
scaling_fractions = ([10] if has_task08_10pct else []) + fractions_run

# 1. Manifest for the 52 Task 09 runs (fractions_run)
order_method = {"dft": 0, "c0": 1, "seq": 2}
manifest_records = []
for key in sorted([k for k in runs_by_key if k[1] in set(fractions_run)], key=lambda k: (k[1], k[0], order_method[k[2]], k[3])):
    direction, frac, method, seed = key
    meta, metrics, cost, comp, _ = runs_by_key[key]
    primary_view = "V3-NEU/none" if direction == "SA" else "V7-VSMEC/none"
    test_m = metrics["test_metrics"][primary_view]
    n_train_sent = int(meta["data_provenance"]["n_train"])
    stage1_steps = ""
    stage1_s = ""
    stage1_h = ""
    total_steps = int(cost["actual_update_steps"])
    total_s = float(cost["train_seconds"])
    total_h = float(cost["gpu_hours"])
    stage1_dft_id = ""
    if method == "c0":
        dft_cost = cost_cache[meta["source_ckpt_provenance"]["stage1_dft_run_id"]]
        stage1_steps = int(dft_cost["actual_update_steps"])
        stage1_s = float(dft_cost["train_seconds"])
        stage1_h = float(dft_cost["gpu_hours"])
        total_steps += stage1_steps
        total_s += stage1_s
        total_h += stage1_h
        stage1_dft_id = meta["source_ckpt_provenance"]["stage1_dft_run_id"]
    manifest_records.append({
        "fraction": frac,
        "n_train_sentences": n_train_sent,
        "direction": direction,
        "method": method,
        "seed": seed,
        "lr": float(meta["lr"]),
        "run_id": meta["run_id"],
        "dev_selection_score": float(metrics["dev_selection_score"]),
        "test_primary_view": primary_view.split("/")[0],
        "test_primary_macro_f1": float(test_m["macro_f1"]),
        "test_primary_accuracy": float(test_m["accuracy"]),
        "best_epoch": metrics["best_epoch"],
        "best_step": int(metrics["best_step"]),
        "early_stopped": bool(metrics["early_stopped"]),
        "stage2_update_steps": int(cost["actual_update_steps"]),
        "stage1_dft_update_steps": stage1_steps,
        "total_target_update_steps": total_steps,
        "stage2_train_seconds": float(cost["train_seconds"]),
        "stage1_dft_train_seconds": stage1_s,
        "total_target_train_seconds": total_s,
        "stage2_gpu_hours": float(cost["gpu_hours"]),
        "total_target_gpu_hours": total_h,
        "peak_vram_mb": float(cost["peak_vram_mb"]),
        "ckpt_saved": bool(comp.get("ckpt_path")),
        "stage1_dft_run_id": stage1_dft_id,
    })

manifest_df = pd.DataFrame(manifest_records)
_atomic_csv(manifest_df, tables_dir / "task09_manifest_52runs.csv")

# 2. Metrics summary table across scaling_fractions (10%, 25%, 50%)
summary_rows = []
for frac in scaling_fractions:
    for direction, method in (("SA", "dft"), ("SA", "c0"), ("SA", "seq"), ("SB", "dft"), ("SB", "c0"), ("SB", "seq")):
        seeds_avail = [0, 1, 2] if method == "c0" else [0, 1, 2, 3, 4]
        n_train_sent = int(runs_by_key[(direction, frac, method, 0)][0]["data_provenance"]["n_train"])
        sub_df = rows_all_df.loc[
            rows_all_df["fraction"].astype(int).eq(frac)
            & rows_all_df["direction"].eq(direction)
            & rows_all_df["method"].eq(method)
            & rows_all_df["split"].eq("test")
        ]
        dev_scores = [float(runs_by_key[(direction, frac, method, s)][1]["dev_selection_score"]) for s in seeds_avail]
        view_order = [("V3-NEU", "none")] if direction == "SA" else [
            ("V7-VSMEC", "none"), ("V5-VSMEC", "none"),
            ("V3-VSMEC", "sum"), ("V3-VSMEC", "argmax"),
            ("V3-VSMEC-strict", "sum"), ("V3-VSMEC-strict", "argmax"),
        ]
        for view, mapping in view_order:
            vsub = sub_df.loc[sub_df["label_view"].eq(view) & sub_df["map_method"].eq(mapping)].sort_values("seed")
            assert vsub["seed"].astype(int).tolist() == seeds_avail, (frac, direction, method, view, mapping)
            row = {
                "direction": direction,
                "fraction": frac,
                "n_train_sentences": n_train_sent,
                "method": method,
                "n_seeds": len(seeds_avail),
                "label_view": view,
                "map_method": mapping,
                "n_eval": int(vsub["n_eval"].iloc[0]),
                "dev_selection_mean": float(np.mean(dev_scores)),
                "dev_selection_sd": float(np.std(dev_scores, ddof=1)) if len(seeds_avail) > 1 else 0.0,
            }
            for stat_col in ("macro_f1", "weighted_f1", "accuracy"):
                vals = vsub[stat_col].astype(float).to_numpy()
                row[f"{stat_col}_mean"] = float(vals.mean())
                row[f"{stat_col}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
            summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
_atomic_csv(summary_df, tables_dir / "task09_metrics_by_fraction.csv")

# 3. Per-class F1 by fraction (strict exports ONLY negative (0) & positive (2), NEVER neutral (1))
def extract_view_arrays(pred: pd.DataFrame, view: str, mapping: str):
    if view == "V3-VSMEC-strict":
        sub = pred.loc[strict_mask(pred["y_true7"].to_numpy())].reset_index(drop=True)
        labels = (0, 2)
        y_true = sub["y_true3"].to_numpy(dtype=int)
        y_pred = sub["y_pred3_argmax" if mapping == "argmax" else "y_pred3_sum"].to_numpy(dtype=int)
        return sub["id"].to_numpy(), y_true, y_pred, labels
    if view == "V7-VSMEC":
        return pred["id"].to_numpy(), pred["y_true7"].to_numpy(dtype=int), pred["y_pred7"].to_numpy(dtype=int), tuple(range(7))
    if view == "V5-VSMEC":
        return pred["id"].to_numpy(), pred["y_true7"].to_numpy(dtype=int), pred["y_pred7"].to_numpy(dtype=int), tuple(range(5))
    if view in ("V3-NEU", "V3-VSMEC"):
        col = "y_pred3_argmax" if mapping == "argmax" else "y_pred3_sum"
        return pred["id"].to_numpy(), pred["y_true3"].to_numpy(dtype=int), pred[col].to_numpy(dtype=int), (0, 1, 2)
    raise ValueError(f"Unsupported view: {view}")

per_class_rows = []
for frac in scaling_fractions:
    for direction, method in (("SA", "dft"), ("SA", "c0"), ("SA", "seq"), ("SB", "dft"), ("SB", "c0"), ("SB", "seq")):
        seeds_avail = [0, 1, 2] if method == "c0" else [0, 1, 2, 3, 4]
        n_train_sent = int(runs_by_key[(direction, frac, method, 0)][0]["data_provenance"]["n_train"])
        view_order = [("V3-NEU", "none")] if direction == "SA" else [
            ("V7-VSMEC", "none"), ("V5-VSMEC", "none"), ("V3-VSMEC", "sum"), ("V3-VSMEC-strict", "sum")
        ]
        for view, mapping in view_order:
            per_seed_f1 = []
            ref_labels = None
            for seed in seeds_avail:
                _, y_true, y_pred, labels = extract_view_arrays(preds_cache[(direction, frac, method, seed)], view, mapping)
                ref_labels = labels
                per_seed_f1.append(f1_score(y_true, y_pred, labels=list(labels), average=None, zero_division=0))
            mat = np.vstack(per_seed_f1)
            for idx_lbl, lbl_id in enumerate(ref_labels):
                lbl_name = EMOTIONS7[lbl_id] if view in ("V7-VSMEC", "V5-VSMEC") else ID_TO_LABEL3[lbl_id]
                vals = mat[:, idx_lbl]
                per_class_rows.append({
                    "direction": direction,
                    "fraction": frac,
                    "n_train_sentences": n_train_sent,
                    "method": method,
                    "label_view": view,
                    "map_method": mapping,
                    "n_seeds": len(seeds_avail),
                    "class_id": int(lbl_id),
                    "class_name": lbl_name,
                    "f1_mean": float(vals.mean()),
                    "f1_sd": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
                })

per_class_df = pd.DataFrame(per_class_rows)
strict_sub = per_class_df.loc[per_class_df["label_view"].eq("V3-VSMEC-strict")]
assert set(strict_sub["class_name"].tolist()) == {"negative", "positive"}, "Strict per-class chỉ được xuất negative và positive!"
_atomic_csv(per_class_df, tables_dir / "task09_per_class_f1_by_fraction.csv")

# 4. Paired bootstrap contrasts across scaling_fractions
stats_cfg = cfg["statistics"]
boot_n = int(ctx.get("boot_override") or stats_cfg["boot"])
boot_seed = int(stats_cfg["boot_seed"])
ci_level = float(stats_cfg["ci"])
sd_thresh = float(stats_cfg["sd_ratio"]["zero_denom_threshold"])

def build_arm(direction: str, frac: int, method: str, seeds: list[int], view: str, mapping: str):
    preds_list = []
    ref_ids = ref_y = ref_labels = None
    for s in seeds:
        ids, y_true, y_pred, labels = extract_view_arrays(preds_cache[(direction, frac, method, s)], view, mapping)
        if ref_ids is None:
            ref_ids, ref_y, ref_labels = ids, y_true, labels
        else:
            assert ids.tolist() == ref_ids.tolist() and np.array_equal(ref_y, y_true)
        preds_list.append(y_pred)
    return {"ids": ref_ids, "y_true": ref_y, "preds": np.vstack(preds_list), "seeds": list(seeds), "labels": ref_labels}

contrast_rows = []
for frac in scaling_fractions:
    per_frac_specs = [
        (f"P2_SA_{frac}" if frac == 10 else f"EXP_SA_{frac}_SEQ_MINUS_DFT", "SA", "seq_minus_dft", "dft", "seq", [0, 1, 2, 3, 4], "V3-NEU", "none", True),
        (f"P2_SB_{frac}" if frac == 10 else f"EXP_SB_{frac}_SEQ_MINUS_DFT", "SB", "seq_minus_dft", "dft", "seq", [0, 1, 2, 3, 4], "V7-VSMEC", "none", True),
        (f"EXP_SA_{frac}_SEQ_MINUS_C0", "SA", "seq_minus_c0", "c0", "seq", [0, 1, 2], "V3-NEU", "none", False),
        (f"EXP_SA_{frac}_C0_MINUS_DFT", "SA", "c0_minus_dft", "dft", "c0", [0, 1, 2], "V3-NEU", "none", False),
        (f"EXP_SB_{frac}_SEQ_MINUS_C0", "SB", "seq_minus_c0", "c0", "seq", [0, 1, 2], "V7-VSMEC", "none", False),
        (f"EXP_SB_{frac}_C0_MINUS_DFT", "SB", "c0_minus_dft", "dft", "c0", [0, 1, 2], "V7-VSMEC", "none", False),
        (f"EXP_SB_{frac}_V5_SEQ_MINUS_DFT", "SB", "seq_minus_dft", "dft", "seq", [0, 1, 2, 3, 4], "V5-VSMEC", "none", False),
        (f"EXP_SB_{frac}_V5_SEQ_MINUS_C0", "SB", "seq_minus_c0", "c0", "seq", [0, 1, 2], "V5-VSMEC", "none", False),
        (f"EXP_SB_{frac}_V3SUM_SEQ_MINUS_DFT", "SB", "seq_minus_dft", "dft", "seq", [0, 1, 2, 3, 4], "V3-VSMEC", "sum", False),
        (f"EXP_SB_{frac}_V3SUM_SEQ_MINUS_C0", "SB", "seq_minus_c0", "c0", "seq", [0, 1, 2], "V3-VSMEC", "sum", False),
        (f"EXP_SB_{frac}_V3STRICT_SEQ_MINUS_DFT", "SB", "seq_minus_dft", "dft", "seq", [0, 1, 2, 3, 4], "V3-VSMEC-strict", "sum", False),
        (f"EXP_SB_{frac}_V3STRICT_SEQ_MINUS_C0", "SB", "seq_minus_c0", "c0", "seq", [0, 1, 2], "V3-VSMEC-strict", "sum", False),
    ]
    for cid, direction, cname, ref_m, new_m, seeds, view, mapping, want_sd_ratio in per_frac_specs:
        ref_arm = build_arm(direction, frac, ref_m, seeds, view, mapping)
        new_arm = build_arm(direction, frac, new_m, seeds, view, mapping)
        labels = ref_arm["labels"]
        est = paired_gain(ref_arm, new_arm, labels, seeds, boot=boot_n, seed=boot_seed, ci=ci_level, stat="macro_f1")
        n_train_sent = int(runs_by_key[(direction, frac, new_m, 0)][0]["data_provenance"]["n_train"])
        delta = float(est["point"])
        ci_lo = float(est["ci_lo"])
        ci_hi = float(est["ci_hi"])
        point_sign = "positive" if delta > 0 else ("negative" if delta < 0 else "zero")
        ci_excl_zero = bool((ci_lo > 0.0) or (ci_hi < 0.0))
        ci_above_zero = bool(ci_lo > 0.0)
        row = {
            "comparison_id": cid,
            "backbone": "ViSoBERT",
            "direction": direction,
            "fraction": frac,
            "n_train_sentences": n_train_sent,
            "contrast": cname,
            "ref_method": ref_m,
            "new_method": new_m,
            "label_view": view,
            "map_method": mapping,
            "stat": "macro_f1",
            "n_seeds": len(seeds),
            "seeds": ",".join(map(str, seeds)),
            "ref_macro_f1": est["base_f1"],
            "new_macro_f1": est["seq_f1"],
            "ref_sd": est["base_std"],
            "new_sd": est["seq_std"],
            "delta_point": delta,
            "ci_lo": ci_lo,
            "ci_hi": ci_hi,
            "point_sign": point_sign,
            "ci_excludes_zero": ci_excl_zero,
            "ci_above_zero": ci_above_zero,
            "se": est["se"],
            "n_seed_pos": est["n_seed_pos"],
            "p_boot": est["p_boot"],
        }
        if want_sd_ratio:
            sd_est = bootstrap_seed_sd_ratio(
                ref_arm, new_arm, labels, seeds, stat="macro_f1",
                boot=boot_n, boot_seed=boot_seed, ci=ci_level, zero_denom_threshold=sd_thresh,
            )
            row.update({
                "sd_ratio_defined": sd_est["defined"],
                "sd_ratio_point": sd_est["point"],
                "sd_ratio_ci_lo": sd_est["ci_lo"],
                "sd_ratio_ci_hi": sd_est["ci_hi"],
                "sd_ratio_n_nan": sd_est["n_nan"],
            })
        else:
            row.update({
                "sd_ratio_defined": np.nan,
                "sd_ratio_point": np.nan,
                "sd_ratio_ci_lo": np.nan,
                "sd_ratio_ci_hi": np.nan,
                "sd_ratio_n_nan": np.nan,
            })
        contrast_rows.append(row)

contrast_rows = apply_phase2_holm(contrast_rows, cfg)
contrasts_df = pd.DataFrame(contrast_rows)
mask_25_50 = contrasts_df["fraction"].isin([25, 50])
assert contrasts_df.loc[mask_25_50, "exploratory"].all(), "25%/50% bắt buộc là exploratory=True!"
assert contrasts_df.loc[mask_25_50, "p_holm"].isna().all(), "25%/50% không được gán p_holm!"
if has_task08_10pct:
    assert int(contrasts_df["primary"].sum()) == 2
_atomic_csv(contrasts_df, tables_dir / "task09_contrasts_scaling.csv")

# 5. Markdown report + Task 10 (100% SA + LLRD/LoRA) GPU-hour projection
total_train_s = float(manifest_df["stage2_train_seconds"].sum())
total_gpu_h = float(manifest_df["stage2_gpu_hours"].sum())
max_vram_sa = float(manifest_df.loc[manifest_df["direction"].eq("SA"), "peak_vram_mb"].max())
max_vram_sb = float(manifest_df.loc[manifest_df["direction"].eq("SB"), "peak_vram_mb"].max())

sa_50_dft_mean_h = float(manifest_df.loc[manifest_df["fraction"].eq(50) & manifest_df["direction"].eq("SA") & manifest_df["method"].eq("dft"), "stage2_gpu_hours"].mean()) if 50 in fractions_run else 0.0
sa_50_seq_mean_h = float(manifest_df.loc[manifest_df["fraction"].eq(50) & manifest_df["direction"].eq("SA") & manifest_df["method"].eq("seq"), "stage2_gpu_hours"].mean()) if 50 in fractions_run else 0.0
est_100pct_sa_h = 2.0 * (5 * sa_50_dft_mean_h + 5 * sa_50_seq_mean_h) if (sa_50_dft_mean_h > 0) else 0.0

lines = [
    "# Task09 — Báo cáo Mở rộng sang 25% và 50% (`52` lượt `main`, `phase2-v1`)",
    "",
    f"- **Commit:** `{env['commit']}`",
    f"- **Profile môi trường:** `{env['profile_id']}` (session `{env['session_id']}`)",
    f"- **GPU / Precision:** `{env['gpu']}` / `bf16` strict",
    f"- **Python / Torch / CUDA:** `{env['python']}` / `{env['torch']}` / `{env['cuda']}`",
    f"- **Tổng `train_seconds` ({len(manifest_df)} lượt Task 09):** `{total_train_s:.2f}s` (`{total_gpu_h:.4f}` GPU-giờ)",
    f"- **Peak VRAM cực đại (Task 09):** SA = `{max_vram_sa:.1f} MiB`, SB = `{max_vram_sb:.1f} MiB`",
    f"- **Ghép mốc `10%` từ Task 08:** `has_task08_10pct={has_task08_10pct}`" + (f" (Task 08 GPU: `{task08_gpu}`)" if task08_gpu else ""),
    "- **Quy ước thống kê (`phase2-v1`):** Gia đình xác nhận chính chỉ gồm 2 so sánh tại `10%` (`Holm m=2`, `α=0.05`); toàn bộ so sánh ở `25%` và `50%` là **thăm dò (`exploratory=True`, `p_holm=NaN`)**.",
    "- **Ghi chú mốc `100%` & biên lưới `SB`:** Kết quả `100%` GĐ1 chỉ làm tham chiếu lịch sử do khác giao thức; `10` lượt `100% SA` chuẩn `phase2-v1` tách sang trước/cùng Task 10. Cờ `grid_edge=True` ở `SB` (`Task 07`) giữ nguyên.",
    "",
    "## 1. Quỹ đạo chuyển giao theo mức dữ liệu & số câu (`10% → 25% → 50%`)",
    "",
    "| Comparison ID | Chiều | Mức % (`n_train`) | Đối chứng | View | Ref F1 | New F1 | `Δ` (95% CI) | Dấu điểm / `CI ≠ 0` | `p_boot` | Vai trò |",
    "|---|---|---:|---|---|---:|---:|---|---|---:|---|",
]
for _, r in contrasts_df.loc[contrasts_df["label_view"].isin(["V3-NEU", "V7-VSMEC", "V5-VSMEC", "V3-VSMEC-strict"])].iterrows():
    if r["contrast"] not in ("seq_minus_dft", "seq_minus_c0", "c0_minus_dft"):
        continue
    if r["label_view"] in ("V5-VSMEC", "V3-VSMEC-strict") and r["contrast"] != "seq_minus_dft":
        continue
    role_txt = f"`primary (p_holm={r['p_holm']:.5f})`" if bool(r["primary"]) else "`exploratory`"
    lines.append(
        f"| `{r['comparison_id']}` | `{r['direction']}` | `{int(r['fraction'])}% ({int(r['n_train_sentences'])})` | `{r['contrast']}` | `{r['label_view']}` | "
        f"`{r['ref_macro_f1']:.4f}` | `{r['new_macro_f1']:.4f}` | "
        f"`{r['delta_point']:+.4f}` (`[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]`) | "
        f"`{r['point_sign']}` / `excl0={bool(r['ci_excludes_zero'])}` | `{r['p_boot']:.5f}` | {role_txt} |"
    )

lines += [
    "",
    "## 2. Tổng hợp điểm `test` theo mức tài nguyên (`mean ± SD`)",
    "",
    "| Chiều | Mức (`n_train`) | Phương pháp | Seeds | View | `Macro-F1` | `Weighted-F1` | `Accuracy` |",
    "|---|---|---|---:|---|---:|---:|---:|",
]
for _, r in summary_df.loc[
    ((summary_df["direction"].eq("SA")) & (summary_df["label_view"].eq("V3-NEU")))
    | ((summary_df["direction"].eq("SB")) & (summary_df["label_view"].isin(["V7-VSMEC", "V5-VSMEC", "V3-VSMEC-strict"])) & (summary_df["map_method"].isin(["none", "sum"])))
].iterrows():
    lines.append(
        f"| `{r['direction']}` | `{int(r['fraction'])}% ({int(r['n_train_sentences'])})` | `{r['method']}` | `{int(r['n_seeds'])}` | `{r['label_view']}/{r['map_method']}` | "
        f"`{r['macro_f1_mean']:.4f} ± {r['macro_f1_sd']:.4f}` | "
        f"`{r['weighted_f1_mean']:.4f} ± {r['weighted_f1_sd']:.4f}` | "
        f"`{r['accuracy_mean']:.4f} ± {r['accuracy_sd']:.4f}` |"
    )

lines += [
    "",
    "## 3. Kiểm định & Dự toán bước tiếp theo (Mốc `100% SA` + Task 10 `LLRD/LoRA`)",
    "",
    f"- **Kiểm định độc lập (Bước 9):** `{len(manifest_df)}` lượt `main` (`{' + '.join(f'{f}%' for f in fractions_run)}`), `{5 * len(manifest_df)}` dòng CSV, khớp `C0` theo `(direction, fraction, seed)`, `SKIP`, tài nguyên đóng băng: **PASS**.",
    "- **Quy tắc `V3-VSMEC-strict` per-class (`AGENTS.md`):** `task09_per_class_f1_by_fraction.csv` chỉ xuất `negative` và `positive`.",
]
if est_100pct_sa_h > 0:
    lines.append(f"- **Dự toán thô cho 10 lượt `100% SA` (`phase2-v1`, 5 DFT + 5 Seq) trên `{env['gpu']}`:** ~`{est_100pct_sa_h:.2f}` GPU-giờ (ước lượng gấp đôi từ tốc độ trung bình `50% SA`).")

report_path = output_dir / "reports" / "phase2" / "task09_scaling_25_50pct.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_text = "\n".join(lines) + "\n"
report_path.write_text(report_text, encoding="utf-8")
print(report_text)
print("Generated 4 tables in:", tables_dir)
'''

run_python(
    analyze_and_report_script.replace(
        "__CTX_JSON__",
        repr(json.dumps({
            "output_dir": str(OUTPUT_DIR),
            "results_dir": str(RESULTS_DIR),
            "env_dir": str(ENV_DIR),
            "selection_path": str(SELECTION_STAGE_PATH),
            "fractions_to_run": list(FRACTIONS_TO_RUN),
            "task08_dir": str(DRIVE_TASK08_DIR),
            "include_task08": INCLUDE_TASK08_10PCT_IN_SCALING,
        })),
    )
)

def sync_and_verify_drive_export(output_dir: Path, results_dir: Path, drive_root: Path, n_expected_runs: int) -> None:
    for tbl in sorted((results_dir / "tables").glob("*.csv")):
        atomic_copy_file(tbl, drive_root / "tables" / tbl.name)
    atomic_copy_file(output_dir / "reports" / "phase2" / "task09_scaling_25_50pct.md", drive_root / "task09_scaling_25_50pct.md")
    atomic_copy_file(results_dir / "phase2_runs.csv", drive_root / "phase2_runs.csv")
    atomic_copy_file(results_dir / "selection_p1_lr.json", drive_root / "selection_p1_lr.json")
    for folder_name in ("environment", "logs"):
        for item in sorted((output_dir / folder_name).rglob("*")):
            if item.is_file() and item.suffix != ".tmp":
                atomic_copy_file(item, drive_root / folder_name / item.relative_to(output_dir / folder_name))

    must_check = [
        (output_dir / "reports" / "phase2" / "task09_scaling_25_50pct.md", drive_root / "task09_scaling_25_50pct.md"),
        (results_dir / "phase2_runs.csv", drive_root / "phase2_runs.csv"),
        (results_dir / "selection_p1_lr.json", drive_root / "selection_p1_lr.json"),
        (output_dir / "environment" / "environment.json", drive_root / "environment" / "environment.json"),
        *[(results_dir / "tables" / f, drive_root / "tables" / f) for f in (
            "task09_manifest_52runs.csv",
            "task09_metrics_by_fraction.csv",
            "task09_per_class_f1_by_fraction.csv",
            "task09_contrasts_scaling.csv",
        )],
    ]
    for src_f, dst_f in must_check:
        assert dst_f.is_file() and sha256_path(dst_f) == sha256_path(src_f), f"Thiếu hoặc lệch SHA-256 trên Drive: {dst_f}"

    d_comp = sorted((drive_root / "runs").glob("*/completion.json"))
    d_prov = sorted((drive_root / "work").glob("*/provenance.json"))
    d_pred = sorted((drive_root / "runs").glob("*/predictions_test.csv"))
    assert len(d_comp) == n_expected_runs and len(d_prov) == n_expected_runs and len(d_pred) == n_expected_runs, (
        len(d_comp), len(d_prov), len(d_pred), n_expected_runs
    )

if DRIVE_SESSION_ROOT is not None:
    sync_and_verify_drive_export(OUTPUT_DIR, RESULTS_DIR, DRIVE_SESSION_ROOT, 26 * len(FRACTIONS_TO_RUN))
    print("Synced & SHA-256 verified all Task09 deliverables on flat Drive:", DRIVE_SESSION_ROOT)


### Hướng dẫn bàn giao từ Google Drive về máy local
1. Đặt báo cáo `/content/drive/MyDrive/Xdomain/Phase2/Task09/task09_scaling_25_50pct.md` vào `reports/phase2/task09_scaling_25_50pct.md`.
2. Đặt 4 bảng trong `tables/` (và `phase2_runs.csv`, `selection_p1_lr.json`, `runs/`, `work/`, `environment/`, `logs/` nếu cần lưu đầy đủ, không cần tải `checkpoints/` nặng ~7,8 GB về máy local) vào `results/phase2/task09/<YYYY-MM-DD>_ViSoBERT_25_50pct_<GPU>/` (giữ nguyên `LF`, không lưu lại CSV bằng Excel).
3. Lưu lại bản notebook [notebooks/phase2_task09_colab.ipynb](file:///c:/Users/ADMIN/PycharmProjects/Xdomain%20Transfer%20Learning/notebooks/phase2_task09_colab.ipynb) đã chạy có output làm bằng chứng thực thi trước khi chuyển sang **10 lượt `100% SA` + Task 10 (`LLRD` / `LoRA`)**.
